In [1]:
!pip install -q "qiskit>=1.3" qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541

In [2]:
QUICK = True   # False = full study (all chips, more shots; several hours)

In [3]:
import math, importlib, time, statistics
from collections import Counter
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.providers import BackendV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

SEED = 1234
DISTANCE = 3
ROUNDS = 3
LOGICAL_BIT = {"Z": 1, "X": 0}   # Z: store |1>, X: store |+>

# ---------- calibration reader ----------
class Calibration:
    def __init__(self, backend):
        self.t = backend.target
        self.two_q = [g for g in ("ecr", "cz", "cx") if g in self.t.operation_names]
        self.n = self.t.num_qubits
        self.qprops = list(self.t.qubit_properties or [None] * self.n)
        known = [p.error for g in self.two_q for p in self.t[g].values()
                 if p is not None and p.error is not None and p.error < 0.5]
        self.default_2q = float(np.median(known)) if known else 0.01
        edges = self.t.build_coupling_map().get_edges()
        self.adj = {q: set() for q in range(self.n)}
        for a, b in edges:
            if self.edge_error(a, b) < 0.5:
                self.adj[a].add(b); self.adj[b].add(a)
        if not any(self.adj.values()):
            for a, b in edges:
                self.adj[a].add(b); self.adj[b].add(a)

    def _gate(self, name, qargs, field):
        p = self.t[name].get(qargs) if name in self.t.operation_names else None
        return getattr(p, field, None) if p is not None else None

    def edge_error(self, a, b):
        v = [self._gate(g, qa, "error") for g in self.two_q for qa in ((a, b), (b, a))]
        v = [x for x in v if x is not None]
        return min(v) if v else self.default_2q

    def edge_duration(self, a, b):
        v = [self._gate(g, qa, "duration") for g in self.two_q for qa in ((a, b), (b, a))]
        v = [x for x in v if x]
        return min(v) if v else 600e-9

    def readout(self, q):
        e = self._gate("measure", (q,), "error")
        return e if e is not None else 0.05

    def meas_duration(self, q):
        d = self._gate("measure", (q,), "duration")
        return d if d else 1.5e-6

    def t1(self, q):
        p = self.qprops[q]
        return p.t1 if (p is not None and p.t1) else 100e-6

    def t2(self, q):
        p = self.qprops[q]
        return p.t2 if (p is not None and p.t2) else 100e-6

# ---------- candidate chains and scoring ----------
def chains(cal, length):
    seen, stack = set(), [[s] for s in cal.adj]
    while stack:
        path = stack.pop()
        if len(path) == length:
            key = tuple(path) if path[0] < path[-1] else tuple(reversed(path))
            if key not in seen:
                seen.add(key)
                yield key
            continue
        for nb in cal.adj[path[-1]]:
            if nb not in path:
                stack.append(path + [nb])

def round_time(cal, chain):
    gates = max(cal.edge_duration(a, b) for a, b in zip(chain, chain[1:]))
    meas = max(cal.meas_duration(q) for q in chain[1::2])
    return 2 * gates + meas

def error_model(cal, chain, basis):
    data, anc = chain[0::2], chain[1::2]
    tr = round_time(cal, chain)
    p_data = []
    for i, q in enumerate(data):
        T = cal.t1(q) if basis == "Z" else cal.t2(q)
        idle = 0.5 * (1 - math.exp(-tr / T))
        gate = 0.0
        if i > 0:
            gate += cal.edge_error(q, anc[i - 1])
        if i < len(anc):
            gate += cal.edge_error(q, anc[i])
        p_data.append(idle + 0.5 * gate)
    p_meas = [cal.readout(a) + 0.5 * (cal.edge_error(data[j], a) + cal.edge_error(data[j + 1], a))
              for j, a in enumerate(anc)]
    clip = lambda p: float(min(max(p, 1e-5), 0.45))
    return [clip(p) for p in p_data], [clip(p) for p in p_meas]

def chain_cost(cal, chain, basis, rounds):
    p_data, p_meas = error_model(cal, chain, basis)
    return rounds * (sum(p_data) + sum(p_meas)) + sum(cal.readout(q) for q in chain[0::2])

def chain_cost_data(cal, chain, basis, rounds):
    p_data, _ = error_model(cal, chain, basis)
    return rounds * sum(p_data) + sum(cal.readout(q) for q in chain[0::2])

# ---------- memory circuit ----------
def build_memory(distance, rounds, basis, logical_bit, inject=None):
    data = QuantumRegister(distance, "data")
    anc = QuantumRegister(distance - 1, "anc")
    syn = [ClassicalRegister(distance - 1, f"s{r}") for r in range(rounds)]
    out = ClassicalRegister(distance, "final")
    qc = QuantumCircuit(data, anc, *syn, out)
    if logical_bit:
        qc.x(data)
    if basis == "X":
        qc.h(data)
    qc.barrier()
    for r in range(rounds):
        if inject is not None and inject[0] == r:
            (qc.x if basis == "Z" else qc.z)(data[inject[1]])
        if basis == "X":
            qc.h(data)
        for j in range(distance - 1):
            qc.cx(data[j], anc[j])
        for j in range(distance - 1):
            qc.cx(data[j + 1], anc[j])
        if basis == "X":
            qc.h(data)
        qc.measure(anc, syn[r])
        qc.reset(anc)
        qc.barrier()
    if basis == "X":
        qc.h(data)
    qc.measure(data, out)
    return qc

def majority_error(counts, logical_bit):
    shots = sum(counts.values())
    wrong = 0
    for key, n in counts.items():
        final = key.split()[0]
        maj = 1 if 2 * final.count("1") > len(final) else 0
        wrong += n * (maj != logical_bit)
    p = wrong / shots
    return p, math.sqrt(max(p * (1 - p), 1e-12) / shots)

# ---------- run one layout (idle decay modelled) ----------
def evaluate(backend, basis, chain, shots, seed, sim):
    qc = build_memory(DISTANCE, ROUNDS, basis, LOGICAL_BIT[basis])
    kw = dict(optimization_level=3, target=backend.target, seed_transpiler=SEED,
              scheduling_method="alap")
    if chain is not None:
        kw["initial_layout"] = list(chain[0::2]) + list(chain[1::2])
    isa = generate_preset_pass_manager(**kw).run(qc)
    used = list(isa.layout.final_index_layout())[: qc.num_qubits]
    ch = [None] * (2 * DISTANCE - 1)
    ch[0::2] = used[:DISTANCE]
    ch[1::2] = used[DISTANCE:]
    counts = sim.run(isa, shots=shots, seed_simulator=seed).result().get_counts()
    p, e = majority_error(counts, LOGICAL_BIT[basis])
    return p, e, tuple(ch)

# ---------- the team: nominate, pilot, verify with fresh shots ----------
def team_vs_qiskit(name, basis, K=3, pilot=3000, final=10000):
    backend = bk[name]
    cal = Calibration(backend)
    sim = AerSimulator(noise_model=NoiseModel.from_backend(backend))
    pool = list(chains(cal, 2 * DISTANCE - 1))
    if not pool:
        raise RuntimeError("no straight chain of this length")
    noms = sorted(pool, key=lambda c: chain_cost(cal, c, basis, ROUNDS))[:K]
    noms += sorted(pool, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))[:K]
    _, _, auto_chain = evaluate(backend, basis, None, 100, 1, sim)
    noms.append(auto_chain)
    seen, cands = set(), []
    for c in noms:
        key = tuple(c) if c[0] < c[-1] else tuple(reversed(c))
        if key not in seen:
            seen.add(key)
            cands.append(tuple(c))
    best = min(cands, key=lambda c: evaluate(backend, basis, c, pilot, 100, sim)[0])
    pw, ew, _ = evaluate(backend, basis, best, final, 999, sim)
    pq, eq, _ = evaluate(backend, basis, None, final, 999, sim)
    return pw, ew, pq, eq

fp = importlib.import_module("qiskit_ibm_runtime.fake_provider")
bk = {}
for name in sorted(dir(fp)):
    if not name.startswith("Fake") or "Provider" in name or name == "FakeBackendV2":
        continue
    try:
        b = getattr(fp, name)()
    except Exception:
        continue
    if isinstance(b, BackendV2) and b.num_qubits >= 5:
        bk[name] = b
print(f"✅ Setup done. {len(bk)} chip snapshots loaded.")

/usr/local/lib/python3.13/dist-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(


✅ Setup done. 67 chip snapshots loaded.


In [15]:
!pip install -q pymatching
import random, pymatching

ROUNDS = 6
SHOTS_RS = 4000
ROLE_CHIPS = ["FakeBrisbane", "FakeOsaka", "FakeKyiv", "FakeSherbrooke", "FakeTorino",
              "FakeFez", "FakeMarrakesh", "FakeKawasaki", "FakeQuebec", "FakeCusco"]
WINDOWS_PER_CHIP = 4
rng = random.Random(2026)            # fixed seed: windows chosen at random, not hand-picked

print("PRE-REGISTERED PREDICTIONS (written before results):")
print("  P1: role choice matters — median error ratio between the two assignments > 1.5×")
print("  P2: data-role features (decay + readout) explain more variance than ancilla features")
print("  P3: the assignment with healthier data qubits wins in > 70 % of clear-cut pairs")
print("  Tested separately for majority-vote and matching decoders.\n")

def check_matrix(d):
    H = np.zeros((d - 1, d), dtype=np.uint8)
    for j in range(d - 1):
        H[j, j] = H[j, j + 1] = 1
    return H

def make_matching(d, rounds, p_data, p_meas):
    w = lambda p: math.log((1 - p) / p)
    return pymatching.Matching.from_check_matrix(
        check_matrix(d), weights=[w(p) for p in p_data], repetitions=rounds + 1,
        timelike_weights=[w(p) for p in p_meas])

def run_roles(backend, cal, sim, basis, chain, shots, seed):
    qc = build_memory(3, ROUNDS, basis, LOGICAL_BIT[basis])
    isa = generate_preset_pass_manager(
        optimization_level=3, target=backend.target, seed_transpiler=SEED, scheduling_method="alap",
        initial_layout=list(chain[0::2]) + list(chain[1::2])).run(qc)
    mem = sim.run(isa, shots=shots, seed_simulator=seed, memory=True).result().get_memory()
    p_data, p_meas = error_model(cal, chain, basis)
    m = make_matching(3, ROUNDS, p_data, p_meas)
    H, bit = check_matrix(3), LOGICAL_BIT[basis]
    wrong_maj = wrong_mwpm = 0
    for s in mem:
        parts = s.split()
        final = np.array([int(c) for c in parts[0][::-1]], dtype=np.uint8)
        raw = [np.array([int(c) for c in p[::-1]], dtype=np.uint8) for p in reversed(parts[1:])]
        S = np.array(raw + [(H @ final) % 2], dtype=np.uint8).T
        D = S.copy()
        D[:, 1:] = S[:, 1:] ^ S[:, :-1]
        corr = m.decode(D)
        wrong_mwpm += int((final ^ corr)[0] != bit)
        wrong_maj += int(int(2 * final.sum() > 3) != bit)
    return wrong_maj / shots, wrong_mwpm / shots

def feats(cal, chain, basis):
    data, anc = chain[0::2], chain[1::2]
    t = ROUNDS * round_time(cal, chain)
    T = cal.t1 if basis == "Z" else cal.t2
    return {"data_decay": sum(1 - math.exp(-t / T(q)) for q in data),
            "data_readout": sum(cal.readout(q) for q in data),
            "anc_decay": sum(1 - math.exp(-t / T(q)) for q in anc),
            "anc_readout": sum(cal.readout(q) for q in anc),
            "gate_error": sum(cal.edge_error(a, b) for a, b in zip(chain, chain[1:]))}

pairs = []
t0 = time.time()
for name in ROLE_CHIPS:
    if name not in bk:
        continue
    backend = bk[name]
    cal = Calibration(backend)
    sim = AerSimulator(noise_model=NoiseModel.from_backend(backend))
    lines = list(chains(cal, 6))
    if not lines:
        continue
    for w in rng.sample(lines, min(WINDOWS_PER_CHIP, len(lines))):
        A, B = tuple(w[0:5]), tuple(w[1:6])          # A: data a,c,e   B: data b,d,f
        for basis in ("Z", "X"):
            try:
                pa = run_roles(backend, cal, sim, basis, A, SHOTS_RS, 31)
                pb = run_roles(backend, cal, sim, basis, B, SHOTS_RS, 31)
            except Exception as e:
                print(f"{name} {w} {basis} skipped ({type(e).__name__})")
                continue
            pairs.append({"chip": name, "basis": basis, "window": w, "pA": pa, "pB": pb,
                          "fA": feats(cal, A, basis), "fB": feats(cal, B, basis)})
            print(f"{name:15s} {basis} window {w}: A(data {A[0::2]}) maj {pa[0]*100:6.2f}% mwpm {pa[1]*100:6.2f}%"
                  f" | B(data {B[0::2]}) maj {pb[0]*100:6.2f}% mwpm {pb[1]*100:6.2f}%  ({(time.time()-t0)/60:.0f} min)")

FLOOR = 1 / SHOTS_RS
KEYS = ["data_decay", "data_readout", "anc_decay", "anc_readout", "gate_error"]

def r2(X, y):
    X1 = np.column_stack([np.ones(len(y)), X])
    coef, *_ = np.linalg.lstsq(X1, y, rcond=None)
    res = y - X1 @ coef
    return 1 - (res @ res) / max(((y - y.mean()) @ (y - y.mean())), 1e-12)

for k_dec, dec in ((0, "MAJORITY VOTE"), (1, "MATCHING DECODER")):
    y = np.array([math.log(max(p["pA"][k_dec], FLOOR) / max(p["pB"][k_dec], FLOOR)) for p in pairs])
    Xall = np.array([[p["fA"][k] - p["fB"][k] for k in KEYS] for p in pairs])
    Xs = (Xall - Xall.mean(0)) / np.where(Xall.std(0) > 0, Xall.std(0), 1)
    idx = {k: i for i, k in enumerate(KEYS)}
    print(f"\n===== {dec}: {len(pairs)} role-swap pairs =====")
    print(f"P1  median error ratio between assignments: ×{math.exp(np.median(np.abs(y))):.2f}")
    print(f"P2  variance explained (R²):  all features {r2(Xs, y):.2f}")
    print(f"      data-role only  (decay + readout)   {r2(Xs[:, [idx['data_decay'], idx['data_readout']]], y):.2f}")
    print(f"      ancilla only    (decay + readout)   {r2(Xs[:, [idx['anc_decay'], idx['anc_readout']]], y):.2f}")
    print(f"      gate errors only                    {r2(Xs[:, [idx['gate_error']]], y):.2f}")
    clear = [(p, yy) for p, yy in zip(pairs, y) if abs(yy) > math.log(1.5)]
    agree = sum(1 for p, yy in clear
                if ((p["fA"]["data_decay"] + p["fA"]["data_readout"]) <
                    (p["fB"]["data_decay"] + p["fB"]["data_readout"])) == (yy < 0))
    print(f"P3  healthier-data assignment wins in {agree}/{len(clear)} clear-cut pairs "
          f"({100*agree/max(len(clear),1):.0f} %)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.9/625.9 kB 2.4 MB/s eta 0:00:00
PRE-REGISTERED PREDICTIONS (written before results):
  P1: role choice matters — median error ratio between the two assignments > 1.5×
  P2: data-role features (decay + readout) explain more variance than ancilla features
  P3: the assignment with healthier data qubits wins in > 70 % of clear-cut pairs
  Tested separately for majority-vote and matching decoders.

FakeBrisbane    Z window (100, 101, 102, 103, 104, 111): A(data (100, 102, 104)) maj   3.10% mwpm   1.68% | B(data (101, 103, 111)) maj   3.12% mwpm   1.80%  (0 min)
FakeBrisbane    X window (100, 101, 102, 103, 104, 111): A(data (100, 102, 104)) maj   5.20% mwpm   3.20% | B(data (101, 103, 111)) maj   2.17% mwpm   1.05%  (0 min)
FakeBrisbane    Z window (55, 68, 67, 66, 73, 85): A(data (55, 67, 73)) maj   0.70% mwpm   0.53% | B(data (68, 66, 85)) maj   6.00% mwpm   4.00%  (0 min)
FakeBrisbane    X window (55, 68, 67, 66, 73, 85): A(data (55, 67, 7

In [4]:
ideal = AerSimulator(seed_simulator=SEED)

def parse(key):
    parts = key.split()
    final = parts[0][::-1]                              # index 0 = data qubit 0
    synd = [p[::-1] for p in reversed(parts[1:])]       # synd[r] = round r
    return final, synd

n_cases = 0
for d in (3, 5):
    for basis in ("Z", "X"):
        for bit in (0, 1):
            for inj in [None] + [(r, k) for r in range(3) for k in range(d)]:
                qc = build_memory(d, 3, basis, bit, inject=inj)
                counts = ideal.run(qc, shots=16).result().get_counts()
                assert len(counts) == 1, f"noiseless run gave several outcomes: {counts}"
                final, synd = parse(next(iter(counts)))
                maj = 1 if 2 * final.count("1") > d else 0
                assert maj == bit, f"majority wrong: d={d} {basis} bit={bit} inject={inj}"
                for r in range(3):
                    flagged = "1" in synd[r]
                    should = inj is not None and r >= inj[0]
                    assert flagged == should, f"syndrome wrong: d={d} {basis} bit={bit} inject={inj} round={r}"
                n_cases += 1
print(f"✅ Self-test passed: {n_cases} noiseless cases, syndromes and majority vote all correct.")

✅ Self-test passed: 104 noiseless cases, syndromes and majority vote all correct.


In [5]:
def best_single_qubit(cal, basis, hold_s):
    def cost(q):
        T = cal.t1(q) if basis == "Z" else cal.t2(q)
        return cal.readout(q) + 0.5 * (1 - math.exp(-hold_s / T))
    return min(range(cal.n), key=cost)

def bare_error(backend, cal, sim, basis, q, hold_s, shots, seed):
    qr, cr = QuantumRegister(1, "q"), ClassicalRegister(1, "final")
    qc = QuantumCircuit(qr, cr)
    if LOGICAL_BIT[basis]:
        qc.x(0)
    if basis == "X":
        qc.h(0)
    qc.delay(hold_s, 0, unit="s")
    if basis == "X":
        qc.h(0)
    qc.measure(0, 0)
    isa = generate_preset_pass_manager(optimization_level=1, target=backend.target, initial_layout=[q],
                                       seed_transpiler=SEED, scheduling_method="alap").run(qc)
    counts = sim.run(isa, shots=shots, seed_simulator=seed).result().get_counts()
    return majority_error(counts, LOGICAL_BIT[basis])

BE_CHIPS = ["FakeOsaka", "FakeTorino", "FakeSherbrooke", "FakeFez"] if QUICK else \
           ["FakeOsaka", "FakeTorino", "FakeSherbrooke", "FakeFez", "FakeKawasaki", "FakeBrisbane",
            "FakeKyiv", "FakeMarrakesh", "FakePrague", "FakeQuebec"]
BE_SHOTS = 10000 if QUICK else 20000
DISTANCE, ROUNDS = 3, 3

print(f"{'chip':15s} basis  {'bare qubit':>18s}  {'3-qubit memory':>18s}  verdict")
for name in BE_CHIPS:
    if name not in bk:
        continue
    backend = bk[name]
    cal = Calibration(backend)
    sim = AerSimulator(noise_model=NoiseModel.from_backend(backend))
    for basis in ("Z", "X"):
        try:
            pool = list(chains(cal, 2 * DISTANCE - 1))
            chain = min(pool, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))
            hold = ROUNDS * round_time(cal, chain)
            q = best_single_qubit(cal, basis, hold)
            pb, eb = bare_error(backend, cal, sim, basis, q, hold, BE_SHOTS, 7)
            pm_, em_, _ = evaluate(backend, basis, chain, BE_SHOTS, 7, sim)
            s = math.hypot(eb, em_)
            v = "BEATS bare" if pb - pm_ > 2*s else ("worse than bare" if pm_ - pb > 2*s else "no sig. difference")
            print(f"{name:15s} {basis}     q{q:<4d} {pb*100:7.3f} %   {pm_*100:7.3f} %          {v}")
        except Exception as e:
            print(f"{name:15s} {basis}  skipped ({type(e).__name__}: {str(e)[:60]})")

chip            basis          bare qubit      3-qubit memory  verdict
FakeOsaka       Z     q63     2.430 %     0.110 %          BEATS bare
FakeOsaka       X     q25     1.390 %     1.230 %          no sig. difference
FakeTorino      Z     q66     2.430 %     0.950 %          BEATS bare
FakeTorino      X     q66     1.420 %     0.080 %          BEATS bare
FakeSherbrooke  Z     q113    2.550 %     0.720 %          BEATS bare
FakeSherbrooke  X     q104    1.380 %     1.340 %          no sig. difference
FakeFez         Z     q2      2.380 %     1.250 %          BEATS bare
FakeFez         X     q56     1.660 %     0.050 %          BEATS bare


In [6]:
if "FakeOsaka" in bk:
    cal = Calibration(bk["FakeOsaka"])
    roles_q = {24: "data", 25: "anc", 26: "data", 27: "anc", 28: "data"}
    roles_c = {25: "data", 26: "anc", 27: "data", 28: "anc", 29: "data"}
    print("qubit | Qiskit role | compiler role |   T1 µs |   T2 µs | readout err")
    for q in range(24, 30):
        print(f"{q:5d} | {roles_q.get(q, '-'):11s} | {roles_c.get(q, '-'):13s} | "
              f"{cal.t1(q)*1e6:7.1f} | {cal.t2(q)*1e6:7.1f} | {cal.readout(q):.4f}")
    print("\nQubits 26 and 28 have very short T2: the compiler keeps them out of the data role.")

qubit | Qiskit role | compiler role |   T1 µs |   T2 µs | readout err
   24 | data        | -             |   316.8 |   175.3 | 0.0058
   25 | anc         | data          |   226.9 |   393.6 | 0.0036
   26 | data        | anc           |   260.1 |    17.7 | 0.0102
   27 | anc         | data          |   337.6 |   136.3 | 0.0117
   28 | data        | anc           |   194.4 |    38.1 | 0.0050
   29 | -           | data          |   353.6 |   282.7 | 0.0078

Qubits 26 and 28 have very short T2: the compiler keeps them out of the data role.


In [7]:
DISTANCE, ROUNDS = 3, 3
SURVEY_CHIPS = ["FakeOsaka", "FakeTorino", "FakeKawasaki", "FakeSherbrooke", "FakeFez", "FakeBrisbane",
                "FakeCusco", "FakeStrasbourg"] if QUICK else list(bk)
PILOT, FINAL = (3000, 10000)

survey = []
t0 = time.time()
for name in SURVEY_CHIPS:
    if name not in bk:
        continue
    for basis in ("Z", "X"):
        try:
            pw, ew, pq, eq = team_vs_qiskit(name, basis, pilot=PILOT, final=FINAL)
        except Exception as e:
            print(f"{name:20s} {basis} skipped ({type(e).__name__}: {str(e)[:50]})")
            continue
        if max(pw, pq) > 0.4:
            print(f"{name:20s} {basis} excluded (error > 40 %: broken snapshot)")
            continue
        s = math.hypot(ew, eq)
        v = "better" if pq - pw > 2*s else ("worse" if pw - pq > 2*s else "same")
        survey.append((name, basis, pw, pq, v))
        print(f"{name:20s} {basis}: team {pw*100:6.3f} %  qiskit {pq*100:6.3f} %  → {v:6s}  ({(time.time()-t0)/60:.0f} min)")

tally = Counter(v for *_, v in survey)
print(f"\nTeam vs Qiskit: better {tally['better']}, same {tally['same']}, worse {tally['worse']} "
      f"({len(survey)} comparisons)")

FakeOsaka            Z: team  0.120 %  qiskit  9.690 %  → better  (0 min)
FakeOsaka            X: team  0.540 %  qiskit  7.320 %  → better  (1 min)
FakeTorino           Z: team  0.920 %  qiskit  2.890 %  → better  (2 min)
FakeTorino           X: team  0.080 %  qiskit  0.140 %  → same    (2 min)
FakeKawasaki         Z: team  0.040 %  qiskit  0.770 %  → better  (3 min)
FakeKawasaki         X: team  0.680 %  qiskit  2.100 %  → better  (3 min)
FakeSherbrooke       Z: team  0.140 %  qiskit  0.150 %  → same    (4 min)
FakeSherbrooke       X: team  0.200 %  qiskit  0.480 %  → better  (4 min)
FakeFez              Z: team  1.060 %  qiskit  2.550 %  → better  (6 min)
FakeFez              X: team  0.010 %  qiskit  0.070 %  → better  (7 min)
FakeBrisbane         Z: team  0.440 %  qiskit  0.480 %  → same    (7 min)
FakeBrisbane         X: team  0.980 %  qiskit  1.400 %  → better  (8 min)
FakeCusco            Z: team  0.920 %  qiskit  6.430 %  → better  (8 min)
FakeCusco            X: team  5.420 % 

In [8]:
ROUNDS = 9
LAMBDA_CHIPS = ["FakeTorino", "FakeSherbrooke"] if QUICK else ["FakeTorino", "FakeKawasaki", "FakeSherbrooke", "FakeBrisbane"]
LAMBDA_FINAL = 10000 if QUICK else 20000

res = {}
t0 = time.time()
for d in (3, 5):
    DISTANCE = d
    for name in LAMBDA_CHIPS:
        for basis in ("Z", "X"):
            try:
                pw, ew, pq, eq = team_vs_qiskit(name, basis, final=LAMBDA_FINAL)
                res[(d, name, basis)] = (pw, pq)
                print(f"d={d} {name:15s} {basis}: team {pw*100:6.3f} %  qiskit {pq*100:6.3f} %  ({(time.time()-t0)/60:.0f} min)")
            except Exception as e:
                print(f"d={d} {name} {basis} skipped ({type(e).__name__}: {str(e)[:60]})")
DISTANCE, ROUNDS = 3, 3

floor = 1 / LAMBDA_FINAL
wins = total = 0
print("\nΛ = error(d=3) / error(d=5)")
for name in LAMBDA_CHIPS:
    for basis in ("Z", "X"):
        if (3, name, basis) in res and (5, name, basis) in res:
            t3, q3 = res[(3, name, basis)]
            t5, q5 = res[(5, name, basis)]
            lt, lq = max(t3, floor) / max(t5, floor), max(q3, floor) / max(q5, floor)
            total += 1
            wins += lt > lq
            print(f"  {name:15s} {basis}: Λ team {lt:6.2f} | Λ qiskit {lq:6.2f}")
print(f"\nTeam has the larger Λ in {wins} of {total} cases.")

d=3 FakeTorino      Z: team  4.720 %  qiskit 13.410 %  (1 min)
d=3 FakeTorino      X: team  0.250 %  qiskit  0.840 %  (2 min)
d=3 FakeSherbrooke  Z: team  0.320 %  qiskit  0.560 %  (2 min)
d=3 FakeSherbrooke  X: team  2.800 %  qiskit  3.530 %  (3 min)
d=5 FakeTorino      Z: team  2.700 %  qiskit  9.860 %  (4 min)
d=5 FakeTorino      X: team  0.060 %  qiskit  0.230 %  (5 min)
d=5 FakeSherbrooke  Z: team  0.150 %  qiskit  0.580 %  (6 min)
d=5 FakeSherbrooke  X: team  0.830 %  qiskit  5.600 %  (7 min)

Λ = error(d=3) / error(d=5)
  FakeTorino      Z: Λ team   1.75 | Λ qiskit   1.36
  FakeTorino      X: Λ team   4.17 | Λ qiskit   3.65
  FakeSherbrooke  Z: Λ team   2.13 | Λ qiskit   0.97
  FakeSherbrooke  X: Λ team   3.37 | Λ qiskit   0.63

Team has the larger Λ in 4 of 4 cases.


In [9]:
DISTANCE, ROUNDS = 3, 9
name, basis = "FakeOsaka", "X"
backend = bk[name]
cal = Calibration(backend)
sim = AerSimulator(noise_model=NoiseModel.from_backend(backend))

def run_with_memory(chain, shots=10000, seed=5):
    qc = build_memory(DISTANCE, ROUNDS, basis, LOGICAL_BIT[basis])
    kw = dict(optimization_level=3, target=backend.target, seed_transpiler=SEED,
              scheduling_method="alap")
    if chain is not None:
        kw["initial_layout"] = list(chain[0::2]) + list(chain[1::2])
    isa = generate_preset_pass_manager(**kw).run(qc)
    used = list(isa.layout.final_index_layout())[: qc.num_qubits]
    mem = sim.run(isa, shots=shots, seed_simulator=seed, memory=True).result().get_memory()
    return used[:DISTANCE], used[DISTANCE:], mem

def diagnose(data_q, anc_q, mem):
    n, d = len(mem), len(data_q)
    flips = np.zeros(d)                 # how often each data qubit ends wrong
    det = np.zeros(d - 1)               # syndrome "alarm" rate per check, per round
    both = 0                            # both checks fire together → middle qubit flipped
    for m in mem:
        parts = m.split()
        final = np.array([int(c) for c in parts[0][::-1]])
        flips += (final != LOGICAL_BIT[basis])
        prev = np.zeros(d - 1, dtype=int)
        for p in reversed(parts[1:]):
            s = np.array([int(c) for c in p[::-1]])
            ev = s ^ prev
            det += ev
            both += int(ev.all())
            prev = s
    print("  data qubit health (measured from the run itself, no calibration used):")
    for k, q in enumerate(data_q):
        print(f"    data q{q:<4d} ends wrong {flips[k]/n*100:6.2f} %   "
              f"[calibration says T2 = {cal.t2(q)*1e6:6.1f} µs]")
    for j, a in enumerate(anc_q):
        print(f"    check via anc q{a:<4d} alarm rate {det[j]/(n*ROUNDS)*100:6.2f} % per round")
    print(f"    both checks at once (middle qubit q{data_q[1]}): {both/(n*ROUNDS)*100:6.2f} % per round")

print("Qiskit's layout:")
dq, aq, mem = run_with_memory(None)
diagnose(dq, aq, mem)

pool = list(chains(cal, 2 * DISTANCE - 1))
team_chain = min(pool, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))
print("\nCompiler's layout:")
dq, aq, mem = run_with_memory(team_chain)
diagnose(dq, aq, mem)

Qiskit's layout:
  data qubit health (measured from the run itself, no calibration used):
    data q24   ends wrong   1.84 %   [calibration says T2 =  175.3 µs]
    data q26   ends wrong  48.75 %   [calibration says T2 =   17.7 µs]
    data q28   ends wrong  40.02 %   [calibration says T2 =   38.1 µs]
    check via anc q25   alarm rate  15.21 % per round
    check via anc q27   alarm rate  20.79 % per round
    both checks at once (middle qubit q26):  10.95 % per round

Compiler's layout:
  data qubit health (measured from the run itself, no calibration used):
    data q25   ends wrong  23.11 %   [calibration says T2 =  393.6 µs]
    data q27   ends wrong  16.42 %   [calibration says T2 =  136.3 µs]
    data q29   ends wrong  10.03 %   [calibration says T2 =  282.7 µs]
    check via anc q26   alarm rate   6.41 % per round
    check via anc q28   alarm rate   5.27 % per round
    both checks at once (middle qubit q27):   1.48 % per round


In [10]:
def syndrome_health(data_q, mem):
    # Per-round flip estimate for each data qubit, from syndromes only (d=3)
    n = len(mem)
    det = np.zeros(2); both = 0
    for m in mem:
        prev = np.zeros(2, dtype=int)
        for p in reversed(m.split()[1:]):
            s = np.array([int(c) for c in p[::-1]])
            ev = s ^ prev
            det += ev
            both += int(ev.all())
            prev = s
    per = n * ROUNDS
    mid = both / per
    return {data_q[0]: det[0] / per - mid, data_q[1]: mid, data_q[2]: det[1] / per - mid}

def logical_from_mem(mem):
    wrong = sum((1 if 2 * m.split()[0].count("1") > DISTANCE else 0) != LOGICAL_BIT[basis] for m in mem)
    return wrong / len(mem)

banned = set()
chain = None                       # start with Qiskit's own layout
for it in range(1, 4):
    dq, aq, mem = run_with_memory(chain, shots=10000, seed=10 + it)
    health = syndrome_health(dq, mem)
    p_log = logical_from_mem(mem)
    worst = max(health, key=health.get)
    print(f"Iteration {it}: data {tuple(dq)}  logical error {p_log*100:6.2f} %")
    print("   syndrome health: " + ", ".join(f"q{q} {r*100:.2f}%/round" for q, r in health.items()))
    banned.add(worst)
    print(f"   → banning q{worst}")
    pool = [c for c in chains(cal, 2 * DISTANCE - 1) if not (set(c) & banned)]
    chain = min(pool, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))
    print(f"   → next chain {chain}\n")

Iteration 1: data (24, 26, 28)  logical error  20.47 %
   syndrome health: q24 4.26%/round, q26 10.94%/round, q28 9.85%/round
   → banning q26
   → next chain (78, 77, 76, 75, 90)

Iteration 2: data (78, 76, 90)  logical error  11.13 %
   syndrome health: q78 27.91%/round, q76 3.59%/round, q90 5.53%/round
   → banning q78
   → next chain (29, 30, 31, 32, 36)

Iteration 3: data (29, 31, 36)  logical error   6.42 %
   syndrome health: q29 12.67%/round, q31 1.69%/round, q36 3.20%/round
   → banning q29
   → next chain (90, 94, 95, 96, 97)



In [11]:
def syndrome_health2(data_q, anc_q, mem):
    # Separate data flips (space-like) from ancilla misreads (same check fires 2 rounds in a row)
    n = len(mem)
    det = np.zeros(2); both = 0; meas = np.zeros(2)
    for m in mem:
        evs, prev = [], np.zeros(2, dtype=int)
        for p in reversed(m.split()[1:]):
            s = np.array([int(c) for c in p[::-1]])
            evs.append(s ^ prev)
            prev = s
        for r, ev in enumerate(evs):
            det += ev
            both += int(ev.all())
            if r + 1 < len(evs):
                for j in (0, 1):
                    o = 1 - j
                    if ev[j] and evs[r+1][j] and not ev[o] and not evs[r+1][o]:
                        meas[j] += 1
    per = n * ROUNDS
    mid = both / per
    data_h = {data_q[0]: max(det[0]/per - mid - 2*meas[0]/per, 0),
              data_q[1]: mid,
              data_q[2]: max(det[1]/per - mid - 2*meas[1]/per, 0)}
    anc_h = {anc_q[0]: meas[0]/per, anc_q[1]: meas[1]/per}
    return data_h, anc_h

def allowed(c, no_data, no_anc):
    return not (set(c[0::2]) & no_data) and not (set(c[1::2]) & no_anc)

# Benchmark: the team's calibration-based pick at the same 9 rounds
pool_all = list(chains(cal, 2 * DISTANCE - 1))
team_chain = min(pool_all, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))
_, _, mem_t = run_with_memory(team_chain, shots=10000, seed=99)
p_team = logical_from_mem(mem_t)
print(f"Benchmark — team's pick {team_chain}: logical error {p_team*100:.2f} %\n")

no_data, no_anc = set(), set()
chain, best = None, (1.0, None)
for it in range(1, 6):
    dq, aq, mem = run_with_memory(chain, shots=10000, seed=10 + it)
    p_log = logical_from_mem(mem)
    err = math.sqrt(max(p_log * (1 - p_log), 1e-12) / len(mem))
    dh, ah = syndrome_health2(dq, aq, mem)
    print(f"Iteration {it}: data {tuple(dq)} anc {tuple(aq)}  logical {p_log*100:6.2f} ± {err*100:.2f} %")
    print("   data health: " + ", ".join(f"q{q} {r*100:.2f}%" for q, r in dh.items())
          + " | ancilla misread: " + ", ".join(f"q{q} {r*100:.2f}%" for q, r in ah.items()))
    if p_log < best[0]:
        best = (p_log, tuple(chain) if chain else tuple(sum(zip(dq, aq + [None]), ())[:-1]))
    elif p_log - best[0] > 2 * err:
        print("   → no improvement: stopping, keeping the best layout found")
        break
    wq, wr = max(dh.items(), key=lambda x: x[1])
    aqw, ar = max(ah.items(), key=lambda x: x[1])
    if ar > wr:
        no_anc.add(aqw); print(f"   → q{aqw} banned from ANCILLA role")
    else:
        no_data.add(wq); print(f"   → q{wq} banned from DATA role (may still serve as ancilla)")
    pool = [c for c in pool_all if allowed(c, no_data, no_anc)]
    chain = min(pool, key=lambda c: chain_cost_data(cal, c, basis, ROUNDS))
    print(f"   → next chain {chain}\n")

print(f"\nSelf-healing best: {best[0]*100:.2f} %  vs  team pick {p_team*100:.2f} %  (started at Qiskit's layout)")

Benchmark — team's pick (25, 26, 27, 28, 29): logical error 6.97 %

Iteration 1: data (24, 26, 28) anc (25, 27)  logical  20.47 ± 0.40 %
   data health: q24 3.03%, q26 10.94%, q28 6.54% | ancilla misread: q25 0.62%, q27 1.65%
   → q26 banned from DATA role (may still serve as ancilla)
   → next chain (25, 26, 27, 28, 29)

Iteration 2: data (25, 27, 29) anc (26, 28)  logical   6.99 ± 0.25 %
   data health: q25 2.54%, q27 1.48%, q29 1.44% | ancilla misread: q26 1.20%, q28 1.18%
   → q25 banned from DATA role (may still serve as ancilla)
   → next chain (78, 77, 76, 75, 90)

Iteration 3: data (78, 76, 90) anc (77, 75)  logical  11.14 ± 0.31 %
   data health: q78 4.67%, q76 3.59%, q90 2.85% | ancilla misread: q77 11.62%, q75 1.34%
   → no improvement: stopping, keeping the best layout found

Self-healing best: 6.99 %  vs  team pick 6.97 %  (started at Qiskit's layout)


In [12]:
DISTANCE, ROUNDS = 3, 9

def run_mem(backend_t, sim_t, basis_, chain, shots=10000, seed=5):
    qc = build_memory(DISTANCE, ROUNDS, basis_, LOGICAL_BIT[basis_])
    kw = dict(optimization_level=3, target=backend_t.target, seed_transpiler=SEED,
              scheduling_method="alap")
    if chain is not None:
        kw["initial_layout"] = list(chain[0::2]) + list(chain[1::2])
    isa = generate_preset_pass_manager(**kw).run(qc)
    used = list(isa.layout.final_index_layout())[: qc.num_qubits]
    mem = sim_t.run(isa, shots=shots, seed_simulator=seed, memory=True).result().get_memory()
    wrong = sum((1 if 2 * m.split()[0].count("1") > DISTANCE else 0) != LOGICAL_BIT[basis_] for m in mem)
    return used[:DISTANCE], used[DISTANCE:], mem, wrong / len(mem)

def self_heal(cal_old, backend_t, sim_t, basis_, start_chain, max_it=5, shots=10000):
    pool = list(chains(cal_old, 2 * DISTANCE - 1))
    no_data, no_anc = set(), set()
    chain, best_p, best_chain, used_it = start_chain, 1.0, start_chain, 0
    for it in range(1, max_it + 1):
        used_it = it
        dq, aq, mem, p = run_mem(backend_t, sim_t, basis_, chain, shots, seed=10 + it)
        err = math.sqrt(max(p * (1 - p), 1e-12) / shots)
        if p < best_p:
            best_p, best_chain = p, chain
        elif p - best_p > 2 * err:
            break
        dh, ah = syndrome_health2(dq, aq, mem)
        wq, wr = max(dh.items(), key=lambda x: x[1])
        aqw, ar = max(ah.items(), key=lambda x: x[1])
        if ar > wr:
            no_anc.add(aqw)
        else:
            no_data.add(wq)
        cand = [c for c in pool if allowed(c, no_data, no_anc)]
        if not cand:
            break
        chain = min(cand, key=lambda c: chain_cost_data(cal_old, c, basis_, ROUNDS))
    return best_chain, used_it

def p_with_err(p, shots=10000):
    return f"{p*100:6.2f} ± {math.sqrt(max(p*(1-p),1e-12)/shots)*100:.2f} %"

STALE, TODAY = "FakeBrisbane", "FakeOsaka"   # yesterday's calibration vs today's real chip
cal_stale = Calibration(bk[STALE])
cal_fresh = Calibration(bk[TODAY])
bt = bk[TODAY]
st = AerSimulator(noise_model=NoiseModel.from_backend(bt))

for basis_ in ("Z", "X"):
    pool_s = list(chains(cal_stale, 2 * DISTANCE - 1))
    pool_f = list(chains(cal_fresh, 2 * DISTANCE - 1))
    stale_pick = min(pool_s, key=lambda c: chain_cost_data(cal_stale, c, basis_, ROUNDS))
    fresh_pick = min(pool_f, key=lambda c: chain_cost_data(cal_fresh, c, basis_, ROUNDS))
    healed, n_it = self_heal(cal_stale, bt, st, basis_, stale_pick)

    FRESH_SEED = 777   # every strategy re-measured with the same fresh shots (no winner's curse)
    p_static = run_mem(bt, st, basis_, stale_pick, seed=FRESH_SEED)[3]
    p_qiskit = run_mem(bt, st, basis_, None, seed=FRESH_SEED)[3]
    p_fresh  = run_mem(bt, st, basis_, fresh_pick, seed=FRESH_SEED)[3]
    p_heal   = run_mem(bt, st, basis_, healed, seed=FRESH_SEED)[3]

    print(f"=== {basis_} basis: calibration from {STALE}, chip behaves like {TODAY}, {ROUNDS} rounds ===")
    print(f"  Static (yesterday's calibration)       {p_with_err(p_static)}   chain {stale_pick}")
    print(f"  Qiskit auto (today's calibration)      {p_with_err(p_qiskit)}")
    print(f"  Compiler (today's calibration)         {p_with_err(p_fresh)}   chain {fresh_pick}")
    print(f"  SELF-HEALING (yesterday's cal + live)  {p_with_err(p_heal)}   chain {healed}  "
          f"after {n_it} run(s)\n")

=== Z basis: calibration from FakeBrisbane, chip behaves like FakeOsaka, 9 rounds ===
  Static (yesterday's calibration)         8.65 ± 0.28 %   chain (10, 11, 12, 17, 30)
  Qiskit auto (today's calibration)       27.84 ± 0.45 %
  Compiler (today's calibration)           2.79 ± 0.16 %   chain (90, 94, 95, 96, 97)
  SELF-HEALING (yesterday's cal + live)    8.65 ± 0.28 %   chain (10, 11, 12, 17, 30)  after 2 run(s)

=== X basis: calibration from FakeBrisbane, chip behaves like FakeOsaka, 9 rounds ===
  Static (yesterday's calibration)        22.35 ± 0.42 %   chain (8, 9, 10, 11, 12)
  Qiskit auto (today's calibration)       20.24 ± 0.40 %
  Compiler (today's calibration)           6.97 ± 0.25 %   chain (25, 26, 27, 28, 29)
  SELF-HEALING (yesterday's cal + live)   13.07 ± 0.34 %   chain (104, 111, 122, 123, 124)  after 3 run(s)



In [13]:
def live_cost(cal_old, c, basis_, live_data):
    p_data, _ = error_model(cal_old, c, basis_)
    pd = [live_data.get(q, p) for q, p in zip(c[0::2], p_data)]
    return ROUNDS * sum(pd) + sum(cal_old.readout(q) for q in c[0::2])

def self_heal_map(cal_old, backend_t, sim_t, basis_, start_chain, budget=6, shots=10000, anc_limit=0.03):
    pool = list(chains(cal_old, 2 * DISTANCE - 1))
    live_data, bad_anc, tried = {}, set(), set()
    chain, best_p, best_chain = start_chain, 1.0, start_chain
    for it in range(1, budget + 1):
        tried.add(tuple(chain))
        dq, aq, mem, p = run_mem(backend_t, sim_t, basis_, chain, shots, seed=20 + it)
        if p < best_p:
            best_p, best_chain = p, chain
        dh, ah = syndrome_health2(dq, aq, mem)
        live_data.update(dh)                                   # today's measured data-role health
        bad_anc |= {q for q, r in ah.items() if r > anc_limit}  # ancillas that misread too often
        print(f"   run {it}: data {tuple(dq)} logical {p*100:6.2f} %  | live: "
              + ", ".join(f"q{q} {r*100:.1f}%" for q, r in dh.items()))
        cand = [c for c in pool if tuple(c) not in tried and not (set(c[1::2]) & bad_anc)]
        if not cand:
            break
        chain = min(cand, key=lambda c: live_cost(cal_old, c, basis_, live_data))
    return best_chain

for basis_ in ("Z", "X"):
    pool_s = list(chains(cal_stale, 2 * DISTANCE - 1))
    stale_pick = min(pool_s, key=lambda c: chain_cost_data(cal_stale, c, basis_, ROUNDS))
    print(f"=== {basis_} basis: self-healing with live health map (yesterday's calibration only) ===")
    healed = self_heal_map(cal_stale, bt, st, basis_, stale_pick)
    p_heal = run_mem(bt, st, basis_, healed, seed=777)[3]
    print(f"   → best chain {healed}, fresh-shot logical error {p_with_err(p_heal)}\n")


=== Z basis: self-healing with live health map (yesterday's calibration only) ===
   run 1: data (10, 12, 30) logical   8.68 %  | live: q10 3.1%, q12 1.7%, q30 3.0%
   run 2: data (13, 17, 29) logical   1.11 %  | live: q13 1.0%, q17 0.8%, q29 2.1%
   run 3: data (84, 92, 103) logical   0.41 %  | live: q84 0.4%, q92 0.3%, q103 0.7%
   run 4: data (92, 103, 111) logical   3.42 %  | live: q92 1.1%, q103 1.2%, q111 11.0%
   run 5: data (86, 84, 92) logical   9.29 %  | live: q86 3.0%, q84 2.8%, q92 0.8%
   run 6: data (61, 63, 65) logical   6.69 %  | live: q61 6.4%, q63 1.8%, q65 1.3%
   → best chain (84, 83, 92, 102, 103), fresh-shot logical error   0.38 ± 0.06 %

=== X basis: self-healing with live health map (yesterday's calibration only) ===
   run 1: data (8, 10, 12) logical  22.44 %  | live: q8 9.6%, q10 3.9%, q12 4.2%
   run 2: data (104, 122, 124) logical  13.17 %  | live: q104 2.7%, q122 2.7%, q124 2.6%
   run 3: data (100, 118, 116) logical  23.29 %  | live: q100 5.4%, q118 11.7%,

In [14]:
def self_heal_map_q(cal_old, backend_t, sim_t, basis_, start_chain, budget=6, shots=3000, anc_limit=0.03):
    pool = list(chains(cal_old, 2 * DISTANCE - 1))
    live_data, bad_anc, tried = {}, set(), set()
    chain, best_p, best_chain = start_chain, 1.0, start_chain
    for it in range(1, budget + 1):
        tried.add(tuple(chain))
        dq, aq, mem, p = run_mem(backend_t, sim_t, basis_, chain, shots, seed=20 + it)
        if p < best_p:
            best_p, best_chain = p, chain
        dh, ah = syndrome_health2(dq, aq, mem)
        live_data.update(dh)
        bad_anc |= {q for q, r in ah.items() if r > anc_limit}
        cand = [c for c in pool if tuple(c) not in tried and not (set(c[1::2]) & bad_anc)]
        if not cand:
            break
        chain = min(cand, key=lambda c: live_cost(cal_old, c, basis_, live_data))
    return best_chain

EAGLES = ["FakeBrisbane", "FakeOsaka", "FakeKyiv", "FakeSherbrooke"]
DISTANCE, ROUNDS = 3, 9
rows = []
t0 = time.time()
for today in EAGLES:
    bt_ = bk[today]
    st_ = AerSimulator(noise_model=NoiseModel.from_backend(bt_))
    cal_f = Calibration(bt_)
    for stale in EAGLES:
        if stale == today:
            continue
        cal_s = Calibration(bk[stale])
        for basis_ in ("Z", "X"):
            try:
                pool_s = list(chains(cal_s, 2 * DISTANCE - 1))
                pool_f = list(chains(cal_f, 2 * DISTANCE - 1))
                stale_pick = min(pool_s, key=lambda c: chain_cost_data(cal_s, c, basis_, ROUNDS))
                fresh_pick = min(pool_f, key=lambda c: chain_cost_data(cal_f, c, basis_, ROUNDS))
                healed = self_heal_map_q(cal_s, bt_, st_, basis_, stale_pick)
                p_static = run_mem(bt_, st_, basis_, stale_pick, seed=777)[3]
                p_fresh  = run_mem(bt_, st_, basis_, fresh_pick, seed=777)[3]
                p_heal   = run_mem(bt_, st_, basis_, healed, seed=777)[3]
                p_team, _, p_qisk, _ = team_vs_qiskit(today, basis_, pilot=3000, final=10000)
                rows.append((stale, today, basis_, p_static, p_qisk, p_fresh, p_team, p_heal))
                print(f"{stale[4:]:>10s}→{today[4:]:<10s} {basis_}: static {p_static*100:6.2f}  qiskit {p_qisk*100:6.2f}  "
                      f"compiler {p_fresh*100:6.2f}  team {p_team*100:6.2f}  HEAL {p_heal*100:6.2f} %"
                      f"  ({(time.time()-t0)/60:.0f} min)")
            except Exception as e:
                print(f"{stale}→{today} {basis_} skipped ({type(e).__name__}: {str(e)[:50]})")

names = ["static", "qiskit (fresh)", "compiler (fresh)", "team (fresh)", "SELF-HEAL (stale+live)"]
print("\nMedian logical error over all cases:")
for k, nm in enumerate(names):
    print(f"  {nm:24s} {statistics.median(r[3+k] for r in rows)*100:6.2f} %")
for k, nm in enumerate(names[:-1]):
    def sig(a, b, n=10000):
        s = math.hypot(math.sqrt(max(a*(1-a),1e-12)/n), math.sqrt(max(b*(1-b),1e-12)/n))
        return "win" if b - a > 2*s else ("loss" if a - b > 2*s else "tie")
    c = Counter(sig(r[7], r[3+k]) for r in rows)
    print(f"  self-heal vs {nm:18s}: win {c['win']}, tie {c['tie']}, loss {c['loss']}")

     Osaka→Brisbane   Z: static  19.50  qiskit   1.90  compiler   4.14  team   1.18  HEAL   6.92 %  (1 min)
     Osaka→Brisbane   X: static   7.92  qiskit   8.94  compiler   6.00  team   5.97  HEAL   5.16 %  (2 min)
      Kyiv→Brisbane   Z: static  14.20  qiskit   1.90  compiler   4.14  team   1.18  HEAL   1.35 %  (3 min)
      Kyiv→Brisbane   X: static  13.42  qiskit   8.94  compiler   6.00  team   5.97  HEAL   6.68 %  (4 min)
Sherbrooke→Brisbane   Z: static   1.97  qiskit   1.90  compiler   4.14  team   1.18  HEAL   1.30 %  (5 min)
Sherbrooke→Brisbane   X: static  12.86  qiskit   8.94  compiler   6.00  team   5.97  HEAL   3.16 %  (6 min)
  Brisbane→Osaka      Z: static   8.65  qiskit  27.88  compiler   2.79  team   0.54  HEAL   0.38 %  (7 min)
  Brisbane→Osaka      X: static  22.35  qiskit  20.02  compiler   6.97  team   6.42  HEAL   2.95 %  (7 min)
      Kyiv→Osaka      Z: static   8.06  qiskit  27.88  compiler   2.79  team   0.54  HEAL   3.17 %  (8 min)
      Kyiv→Osaka      X: sta

In [18]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
token = userdata.get("IBM_QUANTUM_KEY")

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
del token
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=27)
print("Using device:", backend.name)

HW_WINDOWS, HW_SHOTS, ROUNDS = 6, 2000, 6
cal_hw = Calibration(backend)                 # today's live calibration
rng_hw = random.Random(7)                     # windows chosen at random, fixed seed
windows_hw = rng_hw.sample(list(chains(cal_hw, 6)), HW_WINDOWS)

pubs, meta = [], []
for w in windows_hw:
    A, B = tuple(w[0:5]), tuple(w[1:6])
    for basis in ("Z", "X"):
        for tag, ch in (("A", A), ("B", B)):
            qc = build_memory(3, ROUNDS, basis, LOGICAL_BIT[basis])
            isa = generate_preset_pass_manager(
                optimization_level=3, target=backend.target, seed_transpiler=SEED,
                initial_layout=list(ch[0::2]) + list(ch[1::2])).run(qc)
            pubs.append((isa,))
            meta.append({"window": w, "basis": basis, "tag": tag, "chain": ch})

job = Sampler(mode=backend).run(pubs, shots=HW_SHOTS)
print(f"Submitted {len(pubs)} circuits × {HW_SHOTS} shots.")
print("JOB ID (save it!):", job.job_id())

import json
json.dump({"job_id": job.job_id(), "backend": backend.name, "rounds": ROUNDS, "shots": HW_SHOTS,
           "windows": [list(w) for w in windows_hw],
           "meta": [{"window": list(m["window"]), "basis": m["basis"], "tag": m["tag"],
                     "chain": list(m["chain"])} for m in meta]},
          open("hw_job.json", "w"))
print("Saved job details to hw_job.json")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 12:56:50,537: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 12:56:55,033: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 12:56:55,622: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-10-14 12:56:58,244: Using instance: open-instance, plan: open


Using device: ibm_marrakesh


/tmp/ipykernel_861/3594246600.py:27: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = Sampler(mode=backend).run(pubs, shots=HW_SHOTS)


Submitted 24 circuits × 2000 shots.
JOB ID (save it!): db7no7er6oqs7387vjrg
Saved job details to hw_job.json


In [19]:
result = job.result()
H = check_matrix(3)
hw = {}
for res, m in zip(result, meta):
    finals = res.data.final.get_bitstrings()
    synds = [getattr(res.data, f"s{r}").get_bitstrings() for r in range(ROUNDS)]
    p_data, p_meas = error_model(cal_hw, m["chain"], m["basis"])
    mt = make_matching(3, ROUNDS, p_data, p_meas)
    bit = LOGICAL_BIT[m["basis"]]
    n = len(finals)
    wrong_maj = wrong_mwpm = 0
    for i in range(n):
        final = np.array([int(c) for c in finals[i][::-1]], dtype=np.uint8)
        raw = [np.array([int(c) for c in synds[r][i][::-1]], dtype=np.uint8) for r in range(ROUNDS)]
        S = np.array(raw + [(H @ final) % 2], dtype=np.uint8).T
        D = S.copy()
        D[:, 1:] = S[:, 1:] ^ S[:, :-1]
        wrong_mwpm += int((final ^ mt.decode(D))[0] != bit)
        wrong_maj += int(int(2 * final.sum() > 3) != bit)
    hw[(m["window"], m["basis"], m["tag"])] = (wrong_maj / n, wrong_mwpm / n, m["chain"])

print(f"Device {backend.name}, {ROUNDS} rounds, {HW_SHOTS} shots per circuit\n")
ratios = {0: [], 1: []}
calib_right = {0: 0, 1: 0}
clear = {0: 0, 1: 0}
for w in windows_hw:
    for basis in ("Z", "X"):
        a, b = hw[(w, basis, "A")], hw[(w, basis, "B")]
        fA, fB = feats(cal_hw, a[2], basis), feats(cal_hw, b[2], basis)
        calib_says_A = (fA["data_decay"] + fA["data_readout"]) < (fB["data_decay"] + fB["data_readout"])
        print(f"{basis} window {w}: A data {a[2][0::2]} maj {a[0]*100:6.2f}% mwpm {a[1]*100:6.2f}% | "
              f"B data {b[2][0::2]} maj {b[0]*100:6.2f}% mwpm {b[1]*100:6.2f}% | calibration favours {'A' if calib_says_A else 'B'}")
        for k in (0, 1):
            y = math.log(max(a[k], 1 / HW_SHOTS) / max(b[k], 1 / HW_SHOTS))
            ratios[k].append(abs(y))
            if abs(y) > math.log(1.5):
                clear[k] += 1
                calib_right[k] += int(calib_says_A == (y < 0))

for k, name in ((0, "MAJORITY VOTE"), (1, "MATCHING DECODER")):
    print(f"\n{name}: median role-swap error ratio ×{math.exp(np.median(ratios[k])):.2f}  (P1 needs > 1.5)")
    print(f"   calibration picked the better role assignment in {calib_right[k]}/{clear[k]} clear-cut pairs")

Device ibm_marrakesh, 6 rounds, 2000 shots per circuit

Z window (69, 78, 89, 90, 91, 98): A data (69, 89, 91) maj  20.10% mwpm  11.85% | B data (78, 90, 98) maj  20.65% mwpm  11.25% | calibration favours A
X window (69, 78, 89, 90, 91, 98): A data (69, 89, 91) maj   0.35% mwpm   1.10% | B data (78, 90, 98) maj   1.45% mwpm   1.30% | calibration favours A
Z window (122, 123, 124, 125, 126, 127): A data (122, 124, 126) maj  16.20% mwpm   7.55% | B data (123, 125, 127) maj  25.00% mwpm  13.35% | calibration favours A
X window (122, 123, 124, 125, 126, 127): A data (122, 124, 126) maj   0.70% mwpm   2.00% | B data (123, 125, 127) maj   0.55% mwpm   2.35% | calibration favours B
Z window (77, 85, 86, 87, 88, 89): A data (77, 86, 88) maj  35.30% mwpm  23.35% | B data (85, 87, 89) maj  17.30% mwpm   8.85% | calibration favours A
X window (77, 85, 86, 87, 88, 89): A data (77, 86, 88) maj   0.40% mwpm   1.10% | B data (85, 87, 89) maj   0.60% mwpm   0.90% | calibration favours B
Z window (34, 

In [20]:
print("Job:", job.job_id(), "| status:", job.status())

import json
json.dump({"job_id": job.job_id(), "backend": backend.name, "rounds": ROUNDS, "shots": HW_SHOTS,
           "windows": [list(w) for w in windows_hw],
           "meta": [{"window": list(m["window"]), "basis": m["basis"], "tag": m["tag"],
                     "chain": list(m["chain"])} for m in meta]},
          open("hw_job.json", "w"))
print("Saved job details to hw_job.json")

Job: db7no7er6oqs7387vjrg | status: DONE
Saved job details to hw_job.json


In [21]:
!pip install -q pymatching
import pymatching, random, json, os, glob, datetime

def check_matrix(d):
    H = np.zeros((d - 1, d), dtype=np.uint8)
    for j in range(d - 1):
        H[j, j] = H[j, j + 1] = 1
    return H

def make_matching(d, rounds, p_data, p_meas):
    w = lambda p: math.log((1 - p) / p)
    return pymatching.Matching.from_check_matrix(
        check_matrix(d), weights=[w(p) for p in p_data], repetitions=rounds + 1,
        timelike_weights=[w(p) for p in p_meas])

def feats(cal, chain, basis):
    data, anc = chain[0::2], chain[1::2]
    t = ROUNDS * round_time(cal, chain)
    T = cal.t1 if basis == "Z" else cal.t2
    return {"data_decay": sum(1 - math.exp(-t / T(q)) for q in data),
            "data_readout": sum(cal.readout(q) for q in data),
            "anc_decay": sum(1 - math.exp(-t / T(q)) for q in anc),
            "anc_readout": sum(cal.readout(q) for q in anc)}

def snapshot(cal, m):
    p_data, p_meas = error_model(cal, m["chain"], m["basis"])
    return {"window": list(m["window"]), "basis": m["basis"], "tag": m["tag"],
            "chain": list(m["chain"]), "p_data": p_data, "p_meas": p_meas,
            "feats": feats(cal, m["chain"], m["basis"])}

FOLDER = "/content/drive/MyDrive/qmem_hw"
print("✅ Helpers ready")

✅ Helpers ready


In [23]:
import os, json, datetime

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    FOLDER = "/content/drive/MyDrive/qmem_hw"
    where = "Google Drive"
except Exception as e:
    print(f"Drive not available ({type(e).__name__}) — saving locally instead")
    FOLDER = "/content/qmem_hw"
    where = "local Colab storage (download the files below!)"
os.makedirs(FOLDER, exist_ok=True)

stamp = datetime.date.today().isoformat()
rec = {"date": stamp, "job_id": job.job_id(), "backend": backend.name, "rounds": ROUNDS,
       "shots": HW_SHOTS, "meta": [snapshot(cal_hw, m) for m in meta]}
day_file = f"{FOLDER}/day_{stamp}_{job.job_id()}.json"
win_file = f"{FOLDER}/windows.json"
json.dump(rec, open(day_file, "w"))
json.dump({"backend": backend.name, "rounds": ROUNDS, "shots": HW_SHOTS,
           "windows": [list(w) for w in windows_hw]}, open(win_file, "w"))
print(f"✅ Day 1 saved to {where}: {os.listdir(FOLDER)}")

if FOLDER.startswith("/content/qmem_hw"):
    try:
        from google.colab import files
        files.download(day_file)
        files.download(win_file)
    except Exception:
        print("Automatic download failed — use the 📁 Files panel on the left: "
              "open the qmem_hw folder, tap ⋮ next to each file → Download.")

Mounted at /content/drive
✅ Day 1 saved to Google Drive: ['day_2026-10-14_db7no7er6oqs7387vjrg.json', 'windows.json']


In [24]:
!pip install -q stim pymatching "qiskit>=1.3" qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.6 MB/s eta 0:00:00


In [25]:
import stim, pymatching, numpy as np, math, random, importlib, time, statistics
fp = importlib.import_module("qiskit_ibm_runtime.fake_provider")

# ---------- real qubit properties from an IBM calibration snapshot ----------
def qubit_pool(backend_name):
    b = getattr(fp, backend_name)()
    t = b.target
    two_q = [g for g in ("ecr", "cz", "cx") if g in t.operation_names]
    qp = t.qubit_properties or [None] * t.num_qubits
    def prop(name, qargs, field):
        if name not in t.operation_names:
            return None
        p = t[name].get(qargs)
        return getattr(p, field, None) if p is not None else None
    edges = {q: [] for q in range(t.num_qubits)}
    for g in two_q:
        for qargs, p in t[g].items():
            if qargs and len(qargs) == 2 and p is not None and p.error is not None and p.error < 0.5:
                edges[qargs[0]].append(p.error)
                edges[qargs[1]].append(p.error)
    md = [prop("measure", (q,), "duration") for q in range(t.num_qubits)]
    md = [x for x in md if x]
    rd = [prop("reset", (q,), "duration") for q in range(t.num_qubits)]
    rd = [x for x in rd if x]
    t_meas = float(np.median(md)) if md else 1.5e-6
    t_reset = float(np.median(rd)) if rd else 1.0e-6
    pool = []
    for q in range(t.num_qubits):
        ro = prop("measure", (q,), "error")
        sx = prop("sx", (q,), "error")
        T1 = qp[q].t1 if qp[q] is not None else None
        T2 = qp[q].t2 if qp[q] is not None else None
        if not ro or not T1 or not T2 or not edges[q] or ro > 0.3:
            continue                                   # unusable / uncalibrated qubit
        pool.append({"q": q, "ro": ro, "p1": sx if sx else 3e-4,
                     "p2": float(np.mean(edges[q])), "T1": T1, "T2": min(T2, 2 * T1)})
    return pool, t_meas + t_reset

def idle_paulis(qd, t):
    # Pauli-twirled amplitude damping (T1) + dephasing (T2) over an idle time t
    a = 1 - math.exp(-t / qd["T1"])
    b = 1 - math.exp(-t / qd["T2"])
    return a / 4, a / 4, max(b / 2 - a / 4, 0.0)

def noise_params(qd, t_idle):
    return {"ro": qd["ro"], "reset": qd["ro"], "p1": qd["p1"], "p2": qd["p2"],
            "idle": idle_paulis(qd, t_idle)}

# ---------- rotated surface code memory with PER-QUBIT noise ----------
def code_layout(d, basis):
    base = stim.Circuit.generated(f"surface_code:rotated_memory_{basis.lower()}", distance=d, rounds=d)
    allq, measq = set(), set()
    def walk(c):
        for inst in c:
            if isinstance(inst, stim.CircuitRepeatBlock):
                walk(inst.body_copy())
            elif inst.name == "QUBIT_COORDS":
                allq.update(t.value for t in inst.targets_copy())
            elif inst.name == "MR":
                measq.update(t.value for t in inst.targets_copy())
    walk(base)
    return base, sorted(allq - measq), sorted(measq)

def add_noise(c, P, data_q):
    out = stim.Circuit()
    for inst in c:
        if isinstance(inst, stim.CircuitRepeatBlock):
            out.append(stim.CircuitRepeatBlock(inst.repeat_count, add_noise(inst.body_copy(), P, data_q)))
            continue
        name = inst.name
        if name in ("M", "MR", "MX"):
            tg = [t.value for t in inst.targets_copy()]
            if name == "MR":                     # data qubits sit idle while ancillas are measured + reset
                for q in data_q:
                    out.append("PAULI_CHANNEL_1", [q], list(P[q]["idle"]))
            for q in tg:
                out.append("Z_ERROR" if name == "MX" else "X_ERROR", [q], P[q]["ro"])
            out.append(inst)
            if name == "MR":
                for q in tg:
                    out.append("X_ERROR", [q], P[q]["reset"])
        elif name in ("R", "RX"):
            out.append(inst)
            for t in inst.targets_copy():
                out.append("Z_ERROR" if name == "RX" else "X_ERROR", [t.value], P[t.value]["reset"])
        elif name == "H":
            out.append(inst)
            for t in inst.targets_copy():
                out.append("DEPOLARIZE1", [t.value], P[t.value]["p1"])
        elif name == "CX":
            out.append(inst)
            tg = [t.value for t in inst.targets_copy()]
            for a, b in zip(tg[0::2], tg[1::2]):
                out.append("DEPOLARIZE2", [a, b], 0.5 * (P[a]["p2"] + P[b]["p2"]))
        else:
            out.append(inst)
    return out

def logical_error(circ, shots, seed=1):
    dem = circ.detector_error_model(decompose_errors=True)
    m = pymatching.Matching.from_detector_error_model(dem)
    det, obs = circ.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    pred = m.decode_batch(det)
    return float(np.mean(np.any(pred != obs, axis=1)))

def per_round(pL, rounds):
    return 1 - (1 - min(pL, 0.4999)) ** (1 / rounds)

print("✅ Stim tools ready")

✅ Stim tools ready


In [26]:
# ---------- SELF-TEST: our per-qubit noise insertion must reproduce Stim's own noise model ----------
p = 0.003
for basis in ("Z", "X"):
    for d in (3, 5):
        base, dq, mq = code_layout(d, basis)
        zero = {q: {"ro": 0, "reset": 0, "p1": 0, "p2": 0, "idle": (0, 0, 0)} for q in dq + mq}
        e0 = logical_error(add_noise(base, zero, dq), 20000)
        assert e0 == 0.0, f"noiseless circuit gave logical errors ({e0})"
        uni = {q: {"ro": p, "reset": p, "p1": p, "p2": p, "idle": (p/3, p/3, p/3)} for q in dq + mq}
        mine = logical_error(add_noise(base, uni, dq), 200000)
        ref = stim.Circuit.generated(f"surface_code:rotated_memory_{basis.lower()}", distance=d, rounds=d,
                                     after_clifford_depolarization=p, after_reset_flip_probability=p,
                                     before_measure_flip_probability=p, before_round_data_depolarization=p)
        theirs = logical_error(ref, 200000)
        ratio = mine / max(theirs, 1e-9)
        print(f"{basis} d={d}: our noise model {mine*100:.3f} %  vs Stim reference {theirs*100:.3f} %  (ratio {ratio:.2f})")
        assert 0.6 < ratio < 1.6, "noise insertion does not match Stim's reference model"
print("✅ Self-test passed: zero noise → zero errors; uniform noise reproduces Stim's reference model.")

Z d=3: our noise model 0.642 %  vs Stim reference 0.640 %  (ratio 1.00)
Z d=5: our noise model 0.326 %  vs Stim reference 0.344 %  (ratio 0.95)
X d=3: our noise model 0.749 %  vs Stim reference 0.696 %  (ratio 1.08)
X d=5: our noise model 0.370 %  vs Stim reference 0.398 %  (ratio 0.93)
✅ Self-test passed: zero noise → zero errors; uniform noise reproduces Stim's reference model.


In [27]:
# ---------- ROLE-ASSIGNMENT STUDY ----------
QUICK = True
CHIPS = ["FakeTorino", "FakeFez", "FakeKawasaki"] if QUICK else \
        ["FakeTorino", "FakeFez", "FakeKawasaki", "FakeMarrakesh", "FakeBrisbane", "FakeSherbrooke"]
DISTANCES = (3, 5, 7)
SHOTS = {3: 200000, 5: 200000, 7: 100000} if QUICK else {3: 500000, 5: 500000, 7: 300000}
BLIND_SAMPLES = 3 if QUICK else 7

print("PRE-REGISTERED PREDICTIONS:")
print("  Q1: role-aware assignment beats role-blind assignment of the SAME qubits at every distance")
print("  Q2: the gain (blind / aware) GROWS with code distance (expected if the chip is below threshold)")
print("  Q3: role-inverted (worst coherence on data) is clearly worse than role-blind\n")

def idle_total(qd, t):
    return sum(idle_paulis(qd, t))

results = []
t0 = time.time()
for chip in CHIPS:
    pool, t_idle = qubit_pool(chip)
    print(f"--- {chip}: {len(pool)} usable qubits, data idle per round {t_idle*1e6:.2f} µs ---")
    for basis in ("Z", "X"):
        for d in DISTANCES:
            base, dq, mq = code_layout(d, basis)
            n = len(dq) + len(mq)
            if len(pool) < n:
                print(f"  d={d}: needs {n} qubits, chip has {len(pool)} usable — skipped")
                continue
            # the SAME set of qubits for every strategy: the n best overall
            best = sorted(pool, key=lambda x: idle_total(x, t_idle) + x["ro"] + x["p2"])[:n]
            rng = random.Random(100 * d + (basis == "X"))

            def run(data_set, meas_set, seed):
                r = random.Random(seed)
                dpos, mpos = dq[:], mq[:]
                r.shuffle(dpos)
                r.shuffle(mpos)
                P = {}
                for pos, qd in zip(dpos, data_set):
                    P[pos] = noise_params(qd, t_idle)
                for pos, qd in zip(mpos, meas_set):
                    P[pos] = noise_params(qd, t_idle)
                circ = add_noise(base, P, dq)
                return per_round(logical_error(circ, SHOTS[d], seed=seed), d)

            by_coh = sorted(best, key=lambda x: idle_total(x, t_idle))
            aware = run(by_coh[:len(dq)], by_coh[len(dq):], 7)                 # best coherence → data
            inverted = run(by_coh[-len(dq):], by_coh[:-len(dq)], 7)            # worst coherence → data
            blind_vals = []
            for s in range(BLIND_SAMPLES):
                shuffled = best[:]
                rng.shuffle(shuffled)
                blind_vals.append(run(shuffled[:len(dq)], shuffled[len(dq):], 11 + s))
            blind = statistics.median(blind_vals)
            gain = blind / max(aware, 1e-9)
            results.append((chip, basis, d, aware, blind, inverted, gain))
            print(f"  {basis} d={d}: per-round logical error  aware {aware*100:6.3f} %  blind {blind*100:6.3f} %"
                  f"  inverted {inverted*100:6.3f} %   gain ×{gain:5.2f}   ({(time.time()-t0)/60:.0f} min)")

print("\n===== SUMMARY =====")
for basis in ("Z", "X"):
    print(f"{basis} memory:")
    for d in DISTANCES:
        g = [r[6] for r in results if r[1] == basis and r[2] == d]
        a = [r[3] for r in results if r[1] == basis and r[2] == d]
        if g:
            print(f"  d={d}: median gain ×{statistics.median(g):.2f}  (range ×{min(g):.2f}–×{max(g):.2f}), "
                  f"median aware error {statistics.median(a)*100:.3f} %/round")
q1 = sum(r[3] < r[4] for r in results)
q3 = sum(r[5] > r[4] for r in results)
print(f"\nQ1 aware < blind in {q1}/{len(results)} cases;  Q3 inverted > blind in {q3}/{len(results)} cases")
for basis in ("Z", "X"):
    for chip in CHIPS:
        seq = [r[6] for r in results if r[0] == chip and r[1] == basis]
        if len(seq) >= 2:
            trend = "GROWS" if all(b > a for a, b in zip(seq, seq[1:])) else "does not grow steadily"
            print(f"  Q2 {chip} {basis}: gains {['×%.2f' % g for g in seq]} → {trend}")
print("\nIf the aware error does NOT fall as d grows, the chip's noise is near/above threshold for this code,")
print("and the compounding gain (Q2) is not expected — that is itself an informative result.")

PRE-REGISTERED PREDICTIONS:
  Q1: role-aware assignment beats role-blind assignment of the SAME qubits at every distance
  Q2: the gain (blind / aware) GROWS with code distance (expected if the chip is below threshold)
  Q3: role-inverted (worst coherence on data) is clearly worse than role-blind

--- FakeTorino: 129 usable qubits, data idle per round 3.27 µs ---
  Z d=3: per-round logical error  aware  0.876 %  blind  0.954 %  inverted  0.987 %   gain × 1.09   (0 min)
  Z d=5: per-round logical error  aware  0.687 %  blind  0.808 %  inverted  0.893 %   gain × 1.18   (0 min)
  Z d=7: per-round logical error  aware  1.027 %  blind  1.243 %  inverted  1.473 %   gain × 1.21   (0 min)
  X d=3: per-round logical error  aware  0.852 %  blind  1.013 %  inverted  1.277 %   gain × 1.19   (0 min)
  X d=5: per-round logical error  aware  0.710 %  blind  0.946 %  inverted  1.266 %   gain × 1.33   (1 min)
  X d=7: per-round logical error  aware  1.092 %  blind  1.676 %  inverted  2.329 %   gain × 1

In [28]:
print("Λ = error(d) / error(d+2)   (Λ > 1: growing the code helps; Λ < 1: it hurts)\n")
for chip in CHIPS:
    for basis in ("Z", "X"):
        rows = {r[2]: r for r in results if r[0] == chip and r[1] == basis}
        parts = []
        for d1, d2 in ((3, 5), (5, 7)):
            if d1 in rows and d2 in rows:
                la = rows[d1][3] / rows[d2][3]
                lb = rows[d1][4] / rows[d2][4]
                parts.append(f"{d1}→{d2}: aware Λ {la:4.2f} | blind Λ {lb:4.2f}")
        print(f"{chip:13s} {basis}: " + "   ".join(parts))

Λ = error(d) / error(d+2)   (Λ > 1: growing the code helps; Λ < 1: it hurts)

FakeTorino    Z: 3→5: aware Λ 1.28 | blind Λ 1.18   5→7: aware Λ 0.67 | blind Λ 0.65
FakeTorino    X: 3→5: aware Λ 1.20 | blind Λ 1.07   5→7: aware Λ 0.65 | blind Λ 0.56
FakeFez       Z: 3→5: aware Λ 1.59 | blind Λ 1.50   5→7: aware Λ 1.17 | blind Λ 1.10
FakeFez       X: 3→5: aware Λ 1.60 | blind Λ 1.32   5→7: aware Λ 1.00 | blind Λ 0.77
FakeKawasaki  Z: 3→5: aware Λ 0.96 | blind Λ 1.04   5→7: aware Λ 0.67 | blind Λ 0.65
FakeKawasaki  X: 3→5: aware Λ 0.97 | blind Λ 0.85   5→7: aware Λ 0.66 | blind Λ 0.54


In [29]:
def run_assign(base, dq, mq, data_set, meas_set, t_idle, shots, seed, d):
    r = random.Random(seed)
    dpos, mpos = dq[:], mq[:]
    r.shuffle(dpos); r.shuffle(mpos)
    P = {}
    for pos, qd in zip(dpos, data_set):
        P[pos] = noise_params(qd, t_idle)
    for pos, qd in zip(mpos, meas_set):
        P[pos] = noise_params(qd, t_idle)
    return per_round(logical_error(add_noise(base, P, dq), shots, seed=seed), d)

TOPK = 60
A, B = [], []
t0 = time.time()
for chip in CHIPS:
    pool, t_idle = qubit_pool(chip)
    score = lambda x: idle_total(x, t_idle) + x["ro"] + x["p2"]
    top = sorted(pool, key=score)[:TOPK]
    for basis in ("Z", "X"):
        for d in DISTANCES:
            base, dq, mq = code_layout(d, basis)
            n, nd = len(dq) + len(mq), len(dq)
            # --- A: constant quality — sample n qubits (with replacement) from the same top-60 pool ---
            rs = random.Random(500 + d)
            synth = [dict(rs.choice(top)) for _ in range(n)]
            byc = sorted(synth, key=lambda x: idle_total(x, t_idle))
            a_aware = run_assign(base, dq, mq, byc[:nd], byc[nd:], t_idle, SHOTS[d], 7, d)
            rs.shuffle(synth)
            a_blind = run_assign(base, dq, mq, synth[:nd], synth[nd:], t_idle, SHOTS[d], 11, d)
            A.append((chip, basis, d, a_aware, a_blind))
            # --- B: role-aware SELECTION from the whole real chip ---
            if len(pool) >= n:
                data_sel = sorted(pool, key=lambda x: idle_total(x, t_idle))[:nd]
                ids = {x["q"] for x in data_sel}
                meas_sel = sorted([x for x in pool if x["q"] not in ids], key=lambda x: x["ro"] + x["p2"])[:n - nd]
                b_sel = run_assign(base, dq, mq, data_sel, meas_sel, t_idle, SHOTS[d], 7, d)
                prev = next(r for r in results if r[0] == chip and r[1] == basis and r[2] == d)
                B.append((chip, basis, d, b_sel, prev[3], prev[4]))
            print(f"{chip:13s} {basis} d={d} done ({(time.time()-t0)/60:.0f} min)")

print("\n===== A: CONSTANT QUALITY (same top-60 pool at every distance) =====")
print("Λ = error(d)/error(d+2); Λ > 1 means growing the code helps")
for chip in CHIPS:
    for basis in ("Z", "X"):
        rows = {r[2]: r for r in A if r[0] == chip and r[1] == basis}
        parts = [f"{d1}→{d2}: aware Λ {rows[d1][3]/rows[d2][3]:4.2f} | blind Λ {rows[d1][4]/rows[d2][4]:4.2f}"
                 for d1, d2 in ((3, 5), (5, 7)) if d1 in rows and d2 in rows]
        print(f"  {chip:13s} {basis}: " + "   ".join(parts))

print("\n===== B: ROLE-AWARE SELECTION vs earlier strategies (real chip, per-round error) =====")
for chip, basis, d, sel, aware, blind in B:
    print(f"  {chip:13s} {basis} d={d}: selection {sel*100:6.3f} %  | best-set+aware {aware*100:6.3f} %"
          f"  | best-set+blind {blind*100:6.3f} %   → ×{blind/max(sel,1e-9):.2f} vs blind")

FakeTorino    Z d=3 done (0 min)
FakeTorino    Z d=5 done (0 min)
FakeTorino    Z d=7 done (0 min)
FakeTorino    X d=3 done (0 min)
FakeTorino    X d=5 done (0 min)
FakeTorino    X d=7 done (0 min)
FakeFez       Z d=3 done (0 min)
FakeFez       Z d=5 done (1 min)
FakeFez       Z d=7 done (1 min)
FakeFez       X d=3 done (1 min)
FakeFez       X d=5 done (1 min)
FakeFez       X d=7 done (1 min)
FakeKawasaki  Z d=3 done (1 min)
FakeKawasaki  Z d=5 done (1 min)
FakeKawasaki  Z d=7 done (1 min)
FakeKawasaki  X d=3 done (1 min)
FakeKawasaki  X d=5 done (1 min)
FakeKawasaki  X d=7 done (1 min)

===== A: CONSTANT QUALITY (same top-60 pool at every distance) =====
Λ = error(d)/error(d+2); Λ > 1 means growing the code helps
  FakeTorino    Z: 3→5: aware Λ 1.29 | blind Λ 1.53   5→7: aware Λ 1.88 | blind Λ 1.40
  FakeTorino    X: 3→5: aware Λ 1.36 | blind Λ 1.53   5→7: aware Λ 1.80 | blind Λ 1.36
  FakeFez       Z: 3→5: aware Λ 1.58 | blind Λ 1.60   5→7: aware Λ 1.86 | blind Λ 1.61
  FakeFez      

In [30]:
# ---------- C: balanced role-specific selection + simulation-in-the-loop local search ----------
ITERS, EVAL_SHOTS = 25, 30000

def data_cost(x, t, d):      # a data qubit idles every round, joins up to 4 CX per round, is read once at the end
    return idle_total(x, t) + 2 * x["p2"] + x["ro"] / d
def meas_cost(x):            # a measure qubit is read + reset every round and joins up to 4 CX
    return 2 * x["ro"] + 2 * x["p2"] + 2 * x["p1"]

def balanced_select(pool, nd, nm, t, d):
    data = sorted(pool, key=lambda x: data_cost(x, t, d))[:nd]
    ids = {x["q"] for x in data}
    meas = sorted([x for x in pool if x["q"] not in ids], key=meas_cost)[:nm]
    return data, meas

def local_search(base, dq, mq, data, meas, pool, t, d):
    used = {x["q"] for x in data + meas}
    unused = [x for x in pool if x["q"] not in used]
    cur = run_assign(base, dq, mq, data, meas, t, EVAL_SHOTS, 3, d)
    rng = random.Random(42)
    for _ in range(ITERS):
        nd2, nm2, un2 = data[:], meas[:], unused[:]
        kind = rng.random()
        if kind < 0.5:   # replace one of the 3 worst data qubits with one of the 3 best spare/measure candidates
            i = max(rng.sample(range(len(nd2)), 3), key=lambda k: data_cost(nd2[k], t, d))
            cands = sorted([("u", k) for k in range(len(un2))] + [("m", k) for k in range(len(nm2))],
                           key=lambda c: data_cost((un2 if c[0] == "u" else nm2)[c[1]], t, d))[:3]
            src, k = rng.choice(cands)
            if src == "u":
                nd2[i], un2[k] = un2[k], nd2[i]
            else:
                nd2[i], nm2[k] = nm2[k], nd2[i]
        else:            # replace one of the 3 worst measure qubits with one of the 3 best spares
            if not un2:
                continue
            j = max(rng.sample(range(len(nm2)), 3), key=lambda k: meas_cost(nm2[k]))
            k = rng.choice(sorted(range(len(un2)), key=lambda k: meas_cost(un2[k]))[:3])
            nm2[j], un2[k] = un2[k], nm2[j]
        val = run_assign(base, dq, mq, nd2, nm2, t, EVAL_SHOTS, 3, d)
        if val < cur:
            data, meas, unused, cur = nd2, nm2, un2, val
    return data, meas

C = []
t0 = time.time()
for chip in CHIPS:
    pool, t_idle = qubit_pool(chip)
    for basis in ("Z", "X"):
        for d in DISTANCES:
            base, dq, mq = code_layout(d, basis)
            nd, nm = len(dq), len(mq)
            if len(pool) < nd + nm:
                continue
            best = sorted(pool, key=lambda x: idle_total(x, t_idle) + x["ro"] + x["p2"])[:nd + nm]
            byc = sorted(best, key=lambda x: idle_total(x, t_idle))
            bd, bm = balanced_select(pool, nd, nm, t_idle, d)
            sd, sm = local_search(base, dq, mq, bd, bm, pool, t_idle, d)
            FRESH = 999                                            # every strategy re-measured with fresh shots
            e_aware = run_assign(base, dq, mq, byc[:nd], byc[nd:], t_idle, SHOTS[d], FRESH, d)
            e_bal = run_assign(base, dq, mq, bd, bm, t_idle, SHOTS[d], FRESH, d)
            e_srch = run_assign(base, dq, mq, sd, sm, t_idle, SHOTS[d], FRESH, d)
            blind = next(r[4] for r in results if r[0] == chip and r[1] == basis and r[2] == d)
            C.append((chip, basis, d, blind, e_aware, e_bal, e_srch))
            print(f"{chip:13s} {basis} d={d}: blind {blind*100:6.3f}  best-set+aware {e_aware*100:6.3f}  "
                  f"balanced {e_bal*100:6.3f}  balanced+search {e_srch*100:6.3f} %/round"
                  f"   → ×{blind/max(e_srch,1e-9):.2f} vs blind  ({(time.time()-t0)/60:.0f} min)")

print("\nMedian gain over role-blind (blind error / strategy error):")
for k, nm_ in ((4, "best-set + aware roles"), (5, "balanced selection"), (6, "balanced + search")):
    for d in DISTANCES:
        g = [r[3] / max(r[k], 1e-9) for r in C if r[2] == d]
        if g:
            print(f"  {nm_:24s} d={d}: ×{statistics.median(g):.2f}  (min ×{min(g):.2f}, max ×{max(g):.2f})")

FakeTorino    Z d=3: blind  0.954  best-set+aware  0.873  balanced  0.878  balanced+search  0.870 %/round   → ×1.10 vs blind  (0 min)
FakeTorino    Z d=5: blind  0.808  best-set+aware  0.706  balanced  0.632  balanced+search  0.624 %/round   → ×1.30 vs blind  (0 min)
FakeTorino    Z d=7: blind  1.243  best-set+aware  1.031  balanced  0.856  balanced+search  0.810 %/round   → ×1.53 vs blind  (1 min)
FakeTorino    X d=3: blind  1.013  best-set+aware  0.864  balanced  0.896  balanced+search  0.862 %/round   → ×1.18 vs blind  (1 min)
FakeTorino    X d=5: blind  0.946  best-set+aware  0.719  balanced  0.686  balanced+search  0.679 %/round   → ×1.39 vs blind  (1 min)
FakeTorino    X d=7: blind  1.676  best-set+aware  1.189  balanced  1.085  balanced+search  1.080 %/round   → ×1.55 vs blind  (2 min)
FakeFez       Z d=3: blind  0.918  best-set+aware  0.823  balanced  0.807  balanced+search  0.804 %/round   → ×1.14 vs blind  (2 min)
FakeFez       Z d=5: blind  0.612  best-set+aware  0.489  bala

In [31]:
# ---------- STEP 2: ROBUSTNESS — more chips, repeated random placements, more shots ----------
CHIPS_R = ["FakeTorino", "FakeFez", "FakeKawasaki", "FakeMarrakesh", "FakeBrisbane",
           "FakeSherbrooke", "FakeKingston", "FakeAachen"]
SHOTS_R = {3: 200000, 5: 200000, 7: 150000}
N_BLIND, N_BAL = 5, 3
R = []
t0 = time.time()
for chip in CHIPS_R:
    if not hasattr(fp, chip):
        print(f"(skipping {chip}: not available)")
        continue
    pool, t_idle = qubit_pool(chip)
    for basis in ("Z", "X"):
        for d in DISTANCES:
            base, dq, mq = code_layout(d, basis)
            nd, nm = len(dq), len(mq)
            if len(pool) < nd + nm:
                continue
            best = sorted(pool, key=lambda x: idle_total(x, t_idle) + x["ro"] + x["p2"])[:nd + nm]
            rng = random.Random(7 * d + (basis == "X"))
            blind = []
            for s in range(N_BLIND):
                sh = best[:]
                rng.shuffle(sh)
                blind.append(run_assign(base, dq, mq, sh[:nd], sh[nd:], t_idle, SHOTS_R[d], 100 + s, d))
            bd, bm = balanced_select(pool, nd, nm, t_idle, d)
            bal = [run_assign(base, dq, mq, bd, bm, t_idle, SHOTS_R[d], 200 + s, d) for s in range(N_BAL)]
            R.append({"chip": chip, "basis": basis, "d": d, "blind": blind, "bal": bal,
                      "frac": (nd + nm) / len(pool)})
            print(f"{chip:15s} {basis} d={d}: blind {statistics.median(blind)*100:6.3f} "
                  f"[{min(blind)*100:.3f}–{max(blind)*100:.3f}]  balanced {statistics.median(bal)*100:6.3f} "
                  f"[{min(bal)*100:.3f}–{max(bal)*100:.3f}] %/round  uses {100*(nd+nm)/len(pool):.0f}% of chip"
                  f"  ({(time.time()-t0)/60:.0f} min)")

print("\n===== ROBUSTNESS SUMMARY =====")
for d in DISTANCES:
    rows = [r for r in R if r["d"] == d]
    g = [statistics.median(r["blind"]) / statistics.median(r["bal"]) for r in rows]
    dom = sum(max(r["bal"]) < min(r["blind"]) for r in rows)
    if rows:
        print(f"d={d}: median gain ×{statistics.median(g):.2f} (min ×{min(g):.2f}, max ×{max(g):.2f}); "
              f"balanced beat EVERY blind sample in {dom}/{len(rows)} cases")
print("\nΛ(5→7) — does growing the code help? (Λ > 1 = yes)")
for chip in CHIPS_R:
    for basis in ("Z", "X"):
        r5 = next((r for r in R if r["chip"] == chip and r["basis"] == basis and r["d"] == 5), None)
        r7 = next((r for r in R if r["chip"] == chip and r["basis"] == basis and r["d"] == 7), None)
        if r5 and r7:
            lb = statistics.median(r5["blind"]) / statistics.median(r7["blind"])
            la = statistics.median(r5["bal"]) / statistics.median(r7["bal"])
            print(f"  {chip:15s} {basis}: blind Λ {lb:4.2f} | balanced Λ {la:4.2f}   "
                  f"(d=7 uses {100*r7['frac']:.0f}% of usable qubits)")

FakeTorino      Z d=3: blind  0.966 [0.859–1.054]  balanced  0.861 [0.839–0.867] %/round  uses 13% of chip  (0 min)
FakeTorino      Z d=5: blind  0.775 [0.733–0.895]  balanced  0.623 [0.614–0.645] %/round  uses 38% of chip  (0 min)
FakeTorino      Z d=7: blind  1.277 [1.106–1.488]  balanced  0.940 [0.847–1.029] %/round  uses 75% of chip  (1 min)
FakeTorino      X d=3: blind  1.068 [1.011–1.184]  balanced  0.909 [0.908–0.926] %/round  uses 13% of chip  (1 min)
FakeTorino      X d=5: blind  0.953 [0.823–1.043]  balanced  0.693 [0.680–0.715] %/round  uses 38% of chip  (1 min)
FakeTorino      X d=7: blind  1.515 [1.450–1.595]  balanced  1.101 [1.086–1.120] %/round  uses 75% of chip  (2 min)
FakeFez         Z d=3: blind  0.878 [0.825–0.949]  balanced  0.804 [0.790–0.808] %/round  uses 11% of chip  (2 min)
FakeFez         Z d=5: blind  0.607 [0.562–0.641]  balanced  0.435 [0.431–0.454] %/round  uses 32% of chip  (2 min)
FakeFez         Z d=7: blind  0.550 [0.540–0.580]  balanced  0.349 [0.33

In [32]:
# ---------- STEP 3: HARDWARE Λ TEST on ibm_marrakesh (repetition-code memory, d = 3, 5, 7) ----------
import random, json, os, datetime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
TOKEN = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN)
del TOKEN
backend = service.backend("ibm_marrakesh")
cal = Calibration(backend)
ROUNDS, SHOTS_HW = 6, 2000
T_IDLE = 2.5e-6                         # data idle per round while ancillas are measured + reset

def avg_edge(q):
    nb = cal.adj[q]
    return sum(cal.edge_error(q, n) for n in nb) / len(nb) if nb else cal.default_2q
def idle_tot(q):
    a, b = 1 - math.exp(-T_IDLE / cal.t1(q)), 1 - math.exp(-T_IDLE / cal.t2(q))
    return a / 4 + b / 2
def sx_err(q):
    e = cal._gate("sx", (q,), "error")
    return e if e is not None else 3e-4
def dcost(q, d): return idle_tot(q) + 2 * avg_edge(q) + cal.readout(q) / d      # data role
def mcost(q):    return 2 * cal.readout(q) + 2 * avg_edge(q) + 2 * sx_err(q)    # measure role
def chain_cost(c, d):
    return sum(dcost(q, d) if i % 2 == 0 else mcost(q) for i, q in enumerate(c))

def best_chain(d, beam=300):            # beam search over simple paths of length 2d-1
    paths = [[q] for q in cal.adj if cal.adj[q]]
    for _ in range(2 * d - 2):
        new = [p + [n] for p in paths for n in cal.adj[p[-1]] if n not in p]
        new.sort(key=lambda p: chain_cost(p, d))
        paths = new[:beam]
    return tuple(min(paths, key=lambda p: chain_cost(p, d)))

def random_chain(d, rng):               # role-blind baseline: a random simple path
    while True:
        p = [rng.choice([q for q in cal.adj if cal.adj[q]])]
        while len(p) < 2 * d - 1:
            nxt = [n for n in cal.adj[p[-1]] if n not in p]
            if not nxt:
                break
            p.append(rng.choice(nxt))
        if len(p) == 2 * d - 1:
            return tuple(p)

rng = random.Random(2026)
pubs, meta = [], []
for d in (3, 5, 7):
    plan = [("balanced", best_chain(d)), ("qiskit", None)] + \
           [(f"random{k}", random_chain(d, rng)) for k in range(3)]
    for basis in ("Z", "X"):
        for strat, ch in plan:
            qc = build_memory(d, ROUNDS, basis, LOGICAL_BIT[basis])
            kw = dict(optimization_level=3, target=backend.target, seed_transpiler=SEED)
            if ch is not None:
                kw["initial_layout"] = list(ch[0::2]) + list(ch[1::2])
            isa = generate_preset_pass_manager(**kw).run(qc)
            used = list(isa.layout.final_index_layout())[: qc.num_qubits]
            chain = [None] * (2 * d - 1)
            chain[0::2], chain[1::2] = used[:d], used[d:]
            pubs.append((isa,))
            meta.append({"d": d, "basis": basis, "strategy": strat, "chain": chain})

job = Sampler(mode=backend).run(pubs, shots=SHOTS_HW)
print(f"Submitted {len(pubs)} circuits × {SHOTS_HW} shots on {backend.name}: JOB {job.job_id()}")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = "/content/drive/MyDrive/qmem_hw"
except Exception:
    folder = "/content/qmem_hw"
os.makedirs(folder, exist_ok=True)
json.dump({"job_id": job.job_id(), "rounds": ROUNDS, "meta": meta},
          open(f"{folder}/lambda_{datetime.date.today()}_{job.job_id()}.json", "w"))
print("Saved job record to", folder)

qiskit_runtime_service._discover_account:WARNING:2026-10-14 14:14:45,459: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 14:14:49,004: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 14:14:49,005: Using instance: open-instance, plan: open
/tmp/ipykernel_861/1710758062.py:64: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months af

Submitted 30 circuits × 2000 shots on ibm_marrakesh: JOB db7osneil99s73foj5h0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved job record to /content/drive/MyDrive/qmem_hw


In [33]:
# ---------- STEP 3 analysis (run when the job is DONE) ----------
result = job.result()
err = {}
for res, m in zip(result, meta):
    d, R_ = m["d"], ROUNDS
    finals = res.data.final.get_bitstrings()
    synds = [getattr(res.data, f"s{r}").get_bitstrings() for r in range(R_)]
    ch = m["chain"]
    pd = [min(max(idle_tot(q) + avg_edge(q), 1e-4), 0.4) for q in ch[0::2]]
    pm = [min(max(cal.readout(q) + avg_edge(q), 1e-4), 0.4) for q in ch[1::2]]
    mt = make_matching(d, R_, pd, pm)
    H = check_matrix(d)
    bit, n, wm, ww = LOGICAL_BIT[m["basis"]], len(finals), 0, 0
    for i in range(n):
        final = np.array([int(c) for c in finals[i][::-1]], dtype=np.uint8)
        raw = [np.array([int(c) for c in synds[r][i][::-1]], dtype=np.uint8) for r in range(R_)]
        S = np.array(raw + [(H @ final) % 2], dtype=np.uint8).T
        D = S.copy()
        D[:, 1:] = S[:, 1:] ^ S[:, :-1]
        ww += int((final ^ mt.decode(D))[0] != bit)
        wm += int(int(2 * final.sum() > d) != bit)
    err[(d, m["basis"], m["strategy"])] = (wm / n, ww / n)

def strat_err(d, basis, s, k):
    if s == "random":
        return statistics.median(err[(d, basis, f"random{j}")][k] for j in range(3))
    return err[(d, basis, s)][k]

for k, dec in ((1, "MATCHING DECODER"), (0, "MAJORITY VOTE")):
    print(f"\n===== {dec}: logical error after {ROUNDS} rounds on {backend.name} =====")
    for basis in ("Z", "X"):
        for s in ("balanced", "qiskit", "random"):
            e = [strat_err(d, basis, s, k) for d in (3, 5, 7)]
            fl = 1 / SHOTS_HW
            lam = [max(e[0], fl) / max(e[1], fl), max(e[1], fl) / max(e[2], fl)]
            print(f"  {basis} {s:9s}: d3 {e[0]*100:6.2f}%  d5 {e[1]*100:6.2f}%  d7 {e[2]*100:6.2f}%   "
                  f"Λ(3→5) {lam[0]:4.2f}  Λ(5→7) {lam[1]:4.2f}")


===== MATCHING DECODER: logical error after 6 rounds on ibm_marrakesh =====
  Z balanced : d3   8.60%  d5   3.80%  d7   0.75%   Λ(3→5) 2.26  Λ(5→7) 5.07
  Z qiskit   : d3   6.70%  d5   2.05%  d7   1.25%   Λ(3→5) 3.27  Λ(5→7) 1.64
  Z random   : d3   7.70%  d5   2.70%  d7   2.35%   Λ(3→5) 2.85  Λ(5→7) 1.15
  X balanced : d3   2.35%  d5   0.00%  d7   0.00%   Λ(3→5) 47.00  Λ(5→7) 1.00
  X qiskit   : d3   0.85%  d5   0.70%  d7   0.10%   Λ(3→5) 1.21  Λ(5→7) 7.00
  X random   : d3   1.70%  d5   0.50%  d7   0.10%   Λ(3→5) 3.40  Λ(5→7) 5.00

===== MAJORITY VOTE: logical error after 6 rounds on ibm_marrakesh =====
  Z balanced : d3  12.55%  d5  13.60%  d7   1.90%   Λ(3→5) 0.92  Λ(5→7) 7.16
  Z qiskit   : d3  14.65%  d5   4.55%  d7   2.30%   Λ(3→5) 3.22  Λ(5→7) 1.98
  Z random   : d3  14.65%  d5   5.50%  d7  11.15%   Λ(3→5) 2.66  Λ(5→7) 0.49
  X balanced : d3   0.40%  d5   0.10%  d7   0.00%   Λ(3→5) 4.00  Λ(5→7) 2.00
  X qiskit   : d3   0.55%  d5   0.15%  d7   0.15%   Λ(3→5) 3.67  Λ(5→7) 1.00
 

In [34]:
# ---------- T1: NOMINATE + PILOT on 2 real devices ----------
import random, json, os, datetime, statistics, math
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
TOKEN = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN)
del TOKEN
try:
    print("Account usage:", service.usage())
except Exception as e:
    print(f"(usage info not available: {type(e).__name__}) — check the IBM dashboard")

N_DEVICES = 2
ROUNDS = {"Z": 6, "X": 20}           # X needs more rounds to produce measurable errors
DISTS = (3, 5, 7)
PILOT_SHOTS, VERIFY_SHOTS = 1000, 4000
T_IDLE = 2.5e-6

print("PRE-REGISTERED PREDICTIONS:")
print("  H1: team (pilot winner) error ≤ Qiskit's at every distance, both bases")
print("  H2: team Λ(5→7) ≥ Qiskit Λ(5→7)")
print("  H3: the on-device pilot adds value: team ≤ top-1 balanced chain\n")

pool_b = [b for b in service.backends(operational=True, simulator=False) if b.num_qubits >= 127]
pool_b.sort(key=lambda b: (b.name != "ibm_marrakesh", b.status().pending_jobs))
devices = pool_b[:N_DEVICES]
print("Devices:", [b.name for b in devices])

class Scorer:
    def __init__(self, cal):
        self.cal = cal
    def avg_edge(self, q):
        nb = self.cal.adj[q]
        return sum(self.cal.edge_error(q, n) for n in nb) / len(nb) if nb else self.cal.default_2q
    def idle_tot(self, q):
        a = 1 - math.exp(-T_IDLE / self.cal.t1(q))
        b = 1 - math.exp(-T_IDLE / self.cal.t2(q))
        return a / 4 + b / 2
    def sx(self, q):
        e = self.cal._gate("sx", (q,), "error")
        return e if e is not None else 3e-4
    def dcost(self, q, d): return self.idle_tot(q) + 2 * self.avg_edge(q) + self.cal.readout(q) / d
    def mcost(self, q):    return 2 * self.cal.readout(q) + 2 * self.avg_edge(q) + 2 * self.sx(q)
    def cost(self, c, d):  return sum(self.dcost(q, d) if i % 2 == 0 else self.mcost(q) for i, q in enumerate(c))
    def top_chains(self, d, k=3, beam=300):
        paths = [[q] for q in self.cal.adj if self.cal.adj[q]]
        for _ in range(2 * d - 2):
            new = [p + [n] for p in paths for n in self.cal.adj[p[-1]] if n not in p]
            new.sort(key=lambda p: self.cost(p, d))
            paths = new[:beam]
        out, seen = [], set()
        for p in sorted(paths, key=lambda p: self.cost(p, d)):
            key = frozenset(p)
            if key not in seen:
                seen.add(key)
                out.append(tuple(p))
            if len(out) == k:
                break
        return out
    def random_chain(self, d, rng):
        while True:
            p = [rng.choice([q for q in self.cal.adj if self.cal.adj[q]])]
            while len(p) < 2 * d - 1:
                nxt = [n for n in self.cal.adj[p[-1]] if n not in p]
                if not nxt:
                    break
                p.append(rng.choice(nxt))
            if len(p) == 2 * d - 1:
                return tuple(p)
    def weights(self, chain):
        clip = lambda x: float(min(max(x, 1e-4), 0.4))
        return ([clip(self.idle_tot(q) + self.avg_edge(q)) for q in chain[0::2]],
                [clip(self.cal.readout(q) + self.avg_edge(q)) for q in chain[1::2]])

def transpile_for(backend, d, basis, chain):
    qc = build_memory(d, ROUNDS[basis], basis, LOGICAL_BIT[basis])
    kw = dict(optimization_level=3, target=backend.target, seed_transpiler=SEED)
    if chain is not None:
        kw["initial_layout"] = list(chain[0::2]) + list(chain[1::2])
    isa = generate_preset_pass_manager(**kw).run(qc)
    used = list(isa.layout.final_index_layout())[: qc.num_qubits]
    ch = [None] * (2 * d - 1)
    ch[0::2], ch[1::2] = used[:d], used[d:]
    return isa, ch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER = "/content/drive/MyDrive/qmem_hw"
except Exception:
    FOLDER = "/content/qmem_hw"
os.makedirs(FOLDER, exist_ok=True)

scorers, pilot_jobs = {}, {}
rng = random.Random(77)
for dev in devices:
    sc = Scorer(Calibration(dev))
    scorers[dev.name] = sc
    pubs, meta = [], []
    for d in DISTS:
        tops = sc.top_chains(d)
        nominees = [(f"bal{i+1}", c) for i, c in enumerate(tops)] + [("qiskit", None)] + \
                   [(f"rand{k+1}", sc.random_chain(d, rng)) for k in range(2)]
        for basis in ("Z", "X"):
            for label, ch in nominees:
                isa, used = transpile_for(dev, d, basis, ch)
                pd, pm = sc.weights(used)
                pubs.append((isa,))
                meta.append({"device": dev.name, "d": d, "basis": basis, "label": label,
                             "chain": used, "rounds": ROUNDS[basis], "pd": pd, "pm": pm})
    job = Sampler(mode=dev).run(pubs, shots=PILOT_SHOTS)
    pilot_jobs[dev.name] = (job, meta)
    rec = {"stage": "pilot", "device": dev.name, "job_id": job.job_id(), "meta": meta}
    json.dump(rec, open(f"{FOLDER}/tour_pilot_{datetime.date.today()}_{dev.name}_{job.job_id()}.json", "w"))
    print(f"{dev.name}: submitted {len(pubs)} pilot circuits × {PILOT_SHOTS} shots → JOB {job.job_id()}")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 14:22:17,606: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 14:22:20,940: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 14:22:21,322: Loading instance: open-instance, plan: open


Account usage: {'instance_id': '<IBM_INSTANCE_REDACTED>', 'plan_id': '<REDACTED>', 'usage_consumed_seconds': 37, 'usage_consumed_ru': 0.5981, 'usage_period': {'start_time': '2026-09-16T14:22:21.175Z', 'end_time': '2026-10-14T14:22:21.175Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 563}
PRE-REGISTERED PREDICTIONS:
  H1: team (pilot winner) error ≤ Qiskit's at every distance, both bases
  H2: team Λ(5→7) ≥ Qiskit Λ(5→7)
  H3: the on-device pilot adds value: team ≤ top-1 balanced chain

Devices: ['ibm_marrakesh', 'ibm_kingston']
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_861/3460064250.py:111: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = Sampler(mode=dev).run(pubs, shots=PILOT_SHOTS)


ibm_marrakesh: submitted 36 pilot circuits × 1000 shots → JOB db7p0a6il99s73fojd7g
ibm_kingston: submitted 36 pilot circuits × 1000 shots → JOB db7p0aur6oqs73881kug


In [35]:
# ---------- T2: read pilots, pick winners ON THE CHIP, submit fresh verification runs ----------
def decode_err(res, m):
    d, R = m["d"], m["rounds"]
    finals = res.data.final.get_bitstrings()
    synds = [getattr(res.data, f"s{r}").get_bitstrings() for r in range(R)]
    mt, H = make_matching(d, R, m["pd"], m["pm"]), check_matrix(d)
    bit, n, wm, ww = LOGICAL_BIT[m["basis"]], len(finals), 0, 0
    for i in range(n):
        final = np.array([int(c) for c in finals[i][::-1]], dtype=np.uint8)
        raw = [np.array([int(c) for c in synds[r][i][::-1]], dtype=np.uint8) for r in range(R)]
        S = np.array(raw + [(H @ final) % 2], dtype=np.uint8).T
        D = S.copy()
        D[:, 1:] = S[:, 1:] ^ S[:, :-1]
        ww += int((final ^ mt.decode(D))[0] != bit)
        wm += int(int(2 * final.sum() > d) != bit)
    return (ww if m["basis"] == "Z" else wm) / n, n      # pre-registered: matching for Z, majority for X

verify_jobs = {}
for dev in devices:
    job, meta = pilot_jobs[dev.name]
    print(f"Waiting for {dev.name} pilot job {job.job_id()} ...")
    result = job.result()
    pilot = {}
    for res, m in zip(result, meta):
        pilot[(m["d"], m["basis"], m["label"])] = (decode_err(res, m)[0], m)
    pubs, vmeta = [], []
    print(f"\n{dev.name} pilot results (error after the run):")
    for d in DISTS:
        for basis in ("Z", "X"):
            entries = {lab: v for (dd, bb, lab), v in pilot.items() if dd == d and bb == basis}
            line = "  ".join(f"{lab} {e*100:5.2f}%" for lab, (e, _) in sorted(entries.items()))
            win = min(entries, key=lambda k: entries[k][0])
            print(f"  {basis} d={d}: {line}   → winner {win}")
            for role, lab in (("team", win), ("bal1", "bal1"), ("qiskit", "qiskit")):
                m = entries[lab][1]
                ch = None if lab == "qiskit" else m["chain"]
                isa, used = transpile_for(dev, d, basis, ch)
                pd, pm = scorers[dev.name].weights(used)
                pubs.append((isa,))
                vmeta.append({"device": dev.name, "d": d, "basis": basis, "label": role, "picked": lab,
                              "chain": used, "rounds": ROUNDS[basis], "pd": pd, "pm": pm})
    vjob = Sampler(mode=dev).run(pubs, shots=VERIFY_SHOTS)
    verify_jobs[dev.name] = (vjob, vmeta)
    json.dump({"stage": "verify", "device": dev.name, "job_id": vjob.job_id(), "meta": vmeta},
              open(f"{FOLDER}/tour_verify_{datetime.date.today()}_{dev.name}_{vjob.job_id()}.json", "w"))
    print(f"{dev.name}: submitted {len(pubs)} verification circuits × {VERIFY_SHOTS} shots → JOB {vjob.job_id()}\n")

Waiting for ibm_marrakesh pilot job db7p0a6il99s73fojd7g ...

ibm_marrakesh pilot results (error after the run):
  Z d=3: bal1  9.50%  bal2 19.00%  bal3  8.20%  qiskit  7.90%  rand1 28.70%  rand2 16.00%   → winner qiskit
  X d=3: bal1  1.60%  bal2  0.00%  bal3  2.80%  qiskit  2.00%  rand1  0.80%  rand2  7.50%   → winner bal2
  Z d=5: bal1  3.70%  bal2  2.70%  bal3  4.10%  qiskit  1.60%  rand1  1.70%  rand2  4.10%   → winner qiskit
  X d=5: bal1  0.30%  bal2  0.10%  bal3  0.40%  qiskit  0.40%  rand1  0.00%  rand2  0.30%   → winner rand1
  Z d=7: bal1  0.80%  bal2  1.50%  bal3  0.20%  qiskit  1.00%  rand1  6.90%  rand2  4.80%   → winner bal3
  X d=7: bal1  0.00%  bal2  0.50%  bal3  0.10%  qiskit  0.10%  rand1  0.00%  rand2  0.80%   → winner bal1


/tmp/ipykernel_861/1027440742.py:42: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  vjob = Sampler(mode=dev).run(pubs, shots=VERIFY_SHOTS)


ibm_marrakesh: submitted 18 verification circuits × 4000 shots → JOB db7p0ier6oqs73881l90

Waiting for ibm_kingston pilot job db7p0aur6oqs73881kug ...

ibm_kingston pilot results (error after the run):
  Z d=3: bal1  3.30%  bal2  2.30%  bal3  2.70%  qiskit  4.00%  rand1  2.00%  rand2  2.70%   → winner rand1
  X d=3: bal1  0.00%  bal2  0.30%  bal3  0.30%  qiskit  0.20%  rand1  0.40%  rand2  0.40%   → winner bal1
  Z d=5: bal1  3.10%  bal2  0.80%  bal3  0.60%  qiskit  3.90%  rand1  1.30%  rand2  1.20%   → winner bal3
  X d=5: bal1  0.20%  bal2  0.30%  bal3  0.20%  qiskit  0.00%  rand1  0.50%  rand2  0.00%   → winner qiskit
  Z d=7: bal1  0.10%  bal2  0.60%  bal3  0.70%  qiskit  1.10%  rand1  1.40%  rand2  0.20%   → winner bal1
  X d=7: bal1  0.00%  bal2  0.20%  bal3  0.40%  qiskit  0.00%  rand1  0.00%  rand2  0.10%   → winner bal1
ibm_kingston: submitted 18 verification circuits × 4000 shots → JOB db7pn3eil99s73fokkig



In [37]:
# ---------- T3: verification results — team vs top-1 balanced vs Qiskit (fresh shots) ----------
tally = {"H1_win": 0, "H1_tie": 0, "H1_loss": 0, "H3_ok": 0, "H3_n": 0, "H2_ok": 0, "H2_n": 0}
for dev in devices:
    vjob, vmeta = verify_jobs[dev.name]
    print(f"Waiting for {dev.name} verification job {vjob.job_id()} ...")
    result = vjob.result()
    E = {}
    for res, m in zip(result, vmeta):
        e, n = decode_err(res, m)
        E[(m["d"], m["basis"], m["label"])] = (e, math.sqrt(max(e * (1 - e), 1e-12) / n), m["picked"])
    print(f"\n===== {dev.name} (fresh {VERIFY_SHOTS} shots; Z: matching, {ROUNDS['Z']} rounds; X: majority, {ROUNDS['X']} rounds) =====")
    for basis in ("Z", "X"):
        for lab in ("team", "bal1", "qiskit"):
            row = [E[(d, basis, lab)] for d in DISTS]
            fl = 1 / VERIFY_SHOTS
            lam = max(row[1][0], fl) / max(row[2][0], fl)
            extra = f"  (picked: {', '.join(r[2] for r in row)})" if lab == "team" else ""
            print(f"  {basis} {lab:6s}: " + "  ".join(f"d{d} {r[0]*100:5.2f}±{r[1]*100:.2f}%" for d, r in zip(DISTS, row))
                  + f"   Λ(5→7) {lam:5.2f}{extra}")
        for d in DISTS:
            t, q, b = E[(d, basis, "team")], E[(d, basis, "qiskit")], E[(d, basis, "bal1")]
            s = math.hypot(t[1], q[1])
            if q[0] - t[0] > 2 * s:
                tally["H1_win"] += 1
            elif t[0] - q[0] > 2 * s:
                tally["H1_loss"] += 1
            else:
                tally["H1_tie"] += 1
            tally["H3_n"] += 1
            tally["H3_ok"] += int(t[0] <= b[0] + 2 * math.hypot(t[1], b[1]))
        fl = 1 / VERIFY_SHOTS
        lt = max(E[(5, basis, "team")][0], fl) / max(E[(7, basis, "team")][0], fl)
        lq = max(E[(5, basis, "qiskit")][0], fl) / max(E[(7, basis, "qiskit")][0], fl)
        tally["H2_n"] += 1
        tally["H2_ok"] += int(lt >= lq)
    print()

print("===== VERDICT =====")
print(f"H1 team vs Qiskit: better {tally['H1_win']}, tie {tally['H1_tie']}, WORSE {tally['H1_loss']}  "
      f"(holds if WORSE = 0)")
print(f"H2 team Λ(5→7) ≥ Qiskit Λ(5→7): {tally['H2_ok']}/{tally['H2_n']}")
print(f"H3 team no worse than top-1 balanced (pilot adds value or is harmless): {tally['H3_ok']}/{tally['H3_n']}")

Waiting for ibm_marrakesh verification job db7p0ier6oqs73881l90 ...

===== ibm_marrakesh (fresh 4000 shots; Z: matching, 6 rounds; X: majority, 20 rounds) =====
  Z team  : d3  6.85±0.40%  d5  1.57±0.20%  d7  0.90±0.15%   Λ(5→7)  1.75  (picked: qiskit, qiskit, bal3)
  Z bal1  : d3 10.62±0.49%  d5  4.83±0.34%  d7  0.83±0.14%   Λ(5→7)  5.85
  Z qiskit: d3  5.92±0.37%  d5  1.82±0.21%  d7  0.80±0.14%   Λ(5→7)  2.28
  X team  : d3  0.05±0.04%  d5  0.15±0.06%  d7  0.05±0.04%   Λ(5→7)  3.00  (picked: bal2, rand1, bal1)
  X bal1  : d3  1.27±0.18%  d5  0.40±0.10%  d7  0.00±0.00%   Λ(5→7) 16.00
  X qiskit: d3  1.80±0.21%  d5  0.60±0.12%  d7  0.15±0.06%   Λ(5→7)  4.00

Waiting for ibm_kingston verification job db7pn3eil99s73fokkig ...

===== ibm_kingston (fresh 4000 shots; Z: matching, 6 rounds; X: majority, 20 rounds) =====
  Z team  : d3  2.23±0.23%  d5  1.03±0.16%  d7  0.27±0.08%   Λ(5→7)  3.73  (picked: rand1, bal3, bal1)
  Z bal1  : d3  3.00±0.27%  d5  2.55±0.25%  d7  0.27±0.08%   Λ(5→7)  9.